# 🚀 Routing101 - Multi-Signal Video Retrieval on Kaggle

Notebook này được tối ưu hoá để chạy toàn bộ **Web App Tìm Kiếm Video Đa Tín Hiệu (FastAPI + Static UI)** và **Pipeline** trên Kaggle với các bộ dataset:
- **Dataset 1 (Keyframes & Map)**: `https://www.kaggle.com/datasets/nguyenthanghuu/aic2026-dataset` (Chứa `Keyframes`, `map-keyframes`)
- **Dataset 2 (Embeddings & Metadata)**: `https://www.kaggle.com/datasets/lmcu000/rrqbundle` (Chứa `siglib_embed`, `captions`, `caption_embed`, `ocr`, `transcripts`, `transcript_embed`, `summaries`, `summary_embed`, `filtered_object`)
- **Dataset 3 (Videos MP4)**: `https://www.kaggle.com/datasets/lmcu000/degarr` (Chứa toàn bộ video `.mp4` trả lời query)

### ⚙️ Cấu hình bắt buộc trên thanh công cụ Kaggle:
1. **Accelerator**: Chọn **`GPU T4 x2`** (Khuyên dùng - tương thích PyTorch 100% sm_75) hoặc CPU / P100.
2. **Internet**: Chọn `ON`
3. **Data > Input**: Đã thêm đầy đủ 3 dataset trên (đặc biệt `degarr` để xem video và `rrqbundle` cho vector search).

## 📦 BƯỚC 1: Tải Mã Nguồn Repo & Cài đặt Dependencies

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# 0. Luôn đưa cwd về /kaggle/working trước để tránh lỗi 'deleted working directory'
try:
    os.chdir("/kaggle/working")
except Exception:
    pass
%cd /kaggle/working

WORKSPACE_DIR = Path("/kaggle/working/Routing101")
REPO_URL = "https://github.com/hoangducbao/UnoptimalLonx.git"

# 1. Clone mã nguồn Routing101 về Kaggle nếu chưa có, hoặc git pull mới nhất
if not (WORKSPACE_DIR / "backend").exists():
    print(f"📥 Đang tải mã nguồn từ {REPO_URL}...")
    if WORKSPACE_DIR.exists():
        shutil.rmtree(WORKSPACE_DIR, ignore_errors=True)
    !git clone {REPO_URL} /kaggle/working/Routing101
else:
    print("✅ Mã nguồn Routing101 đã có sẵn trong /kaggle/working/Routing101, đang đồng bộ mới nhất...")
    !cd /kaggle/working/Routing101 && git pull origin main

# 2. Chuyển vào thư mục repository và thiết lập sys.path
%cd /kaggle/working/Routing101
try:
    os.chdir("/kaggle/working/Routing101")
except Exception:
    pass

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# 3. Ghi đè trực tiếp backend/models.py tương thích GPU T4/P100/CPU an toàn
models_py_code = '''import numpy as np
import torch
from PIL import Image
from . import config

def _select_device() -> str:
    if torch.cuda.is_available():
        try:
            cap = torch.cuda.get_device_capability()
            if cap[0] >= 7:
                torch.zeros(1, device="cuda")
                return "cuda"
            else:
                print(f"[Device Warning] GPU compute capability {cap[0]}.{cap[1]} < 7.0 (e.g. Tesla P100). Falling back to CPU.")
                return "cpu"
        except Exception as e:
            print(f"[Device Warning] CUDA check failed ({e}). Falling back to CPU.")
            return "cpu"
    return "cpu"

DEVICE = _select_device()
_siglip2 = None

def load_siglip2():
    global _siglip2
    if _siglip2 is None:
        from transformers import AutoModel, AutoProcessor
        model = AutoModel.from_pretrained(config.SIGLIP2_MODEL_ID).to(DEVICE).eval()
        processor = AutoProcessor.from_pretrained(config.SIGLIP2_MODEL_ID)
        _siglip2 = (model, processor)
    return _siglip2

def encode_text_siglip2(texts: list) -> np.ndarray:
    model, processor = load_siglip2()
    inputs = processor(text=texts, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_text_features(**inputs)
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.float().cpu().numpy().astype("float32")

def encode_image_siglip2(images: list) -> np.ndarray:
    model, processor = load_siglip2()
    inputs = processor(images=images, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.get_image_features(**inputs)
    feats = out.pooler_output if hasattr(out, "pooler_output") else out
    return feats.float().cpu().numpy().astype("float32")

def is_image_query(query) -> bool:
    return isinstance(query, Image.Image)

def siglip2_query_vec(query) -> np.ndarray:
    if is_image_query(query):
        return encode_image_siglip2([query])[0]
    return encode_text_siglip2([query])[0]
'''
(WORKSPACE_DIR / "backend/models.py").write_text(models_py_code, encoding="utf-8")

# 4. Ghi đè trực tiếp backend/main.py hoàn chỉnh
main_py_code = '''from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles

from . import config
from .es_indexing import ensure_all_fuzzy_indices
from .models import DEVICE, load_siglip2
from .routes import export, facets, hierarchy, neighbors, playback, query_image, search, trake
from .search import asr as asr_mod
from .search import caption as cap_mod
from .search import keyframe as kf
from .search import summary as sum_mod

@asynccontextmanager
async def lifespan(app: FastAPI):
    config.tune_thread_pools(DEVICE)
    print(f"[startup] device={DEVICE} cpu_budget={config.CPU_BUDGET}")

    print("[startup] loading SigLIP2 text/image tower…")
    load_siglip2()

    print("[startup] Keyframe — SigLIP2 frame index")
    kf.build_frame_index(config.FRAME_SIGLIP2_GLOB)
    print("[startup] Keyframe — CLIP frame index")
    kf.build_frame_index(config.FRAME_CLIP_GLOB)

    print("[startup] ASR — SigLIP2 index")
    asr_mod.build_siglip_asr_index()
    print("[startup] Caption — SigLIP2 index")
    cap_mod.build_siglip_caption_index()
    print("[startup] Summary — embeddings + SigLIP2 index")
    sum_mod.build_siglip_summary_index()

    try:
        print("[startup] ASR/Caption/OCR/Summary — Elasticsearch")
        ensure_all_fuzzy_indices()
    except Exception as e:
        print(f"[startup] Elasticsearch warning: {e}")

    print("[startup] all signals ready")
    yield

app = FastAPI(title="Routing101 by MiLF", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

app.include_router(search.router)
app.include_router(facets.router)
app.include_router(neighbors.router)
app.include_router(playback.router)
app.include_router(query_image.router)
app.include_router(trake.router)
app.include_router(hierarchy.router)
app.include_router(export.router)

config.THUMBNAIL_ROOT.mkdir(parents=True, exist_ok=True)
config.VIDEO_DIR.mkdir(parents=True, exist_ok=True)
app.mount("/media/keyframes", StaticFiles(directory=config.THUMBNAIL_ROOT), name="keyframes")
app.mount("/media/video", StaticFiles(directory=config.VIDEO_DIR), name="video")
app.mount("/app", StaticFiles(directory=config.REPO_ROOT / "frontend", html=True), name="frontend")

@app.get("/")
def root():
    from fastapi.responses import RedirectResponse
    return RedirectResponse("/app/")
'''
(WORKSPACE_DIR / "backend/main.py").write_text(main_py_code, encoding="utf-8")

# 5. Ghi đè trực tiếp backend/es_indexing.py an toàn tuyệt đối
es_indexing_code = '''import pandas as pd
from . import config
from .es_client import get_es_client

def ensure_asr_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_ASR): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_ASR, mappings={"properties": {"video_id": {"type": "keyword"}, "segment_id": {"type": "integer"}, "start_sec": {"type": "float"}, "text": {"type": "text"}}})
        def _docs():
            if not config.TRANSCRIPTS_DIR.exists(): return
            for csv_path in sorted(config.TRANSCRIPTS_DIR.glob("*.csv")):
                if csv_path.name == "manifest.csv": continue
                df = pd.read_csv(csv_path)
                if df.empty: continue
                video_id = csv_path.stem
                for _, r in df.iterrows():
                    text = r.get("text") or r.get("transcript") or ""
                    yield {"_index": config.ES_INDEX_ASR, "_id": f"{video_id}_{int(r.get('segment_id', 0))}", "_source": {"video_id": video_id, "segment_id": int(r.get("segment_id", 0)), "start_sec": float(r.get("start_sec", 0.0)), "text": str(text)}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES ASR Info] {e}")
        return False

def ensure_caption_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_CAPTION): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_CAPTION, mappings={"properties": {"video_id": {"type": "keyword"}, "frame_id": {"type": "integer"}, "text": {"type": "text"}}})
        def _docs():
            if not config.CAPTIONING_DIR.exists(): return
            for csv_path in sorted(config.CAPTIONING_DIR.glob("*.csv")):
                if csv_path.name == "manifest.csv": continue
                df = pd.read_csv(csv_path)
                if df.empty: continue
                video_id = csv_path.stem
                for _, r in df.iterrows():
                    v_id = r.get("video_id", video_id)
                    text = r.get("caption_text") or r.get("text") or r.get("caption") or ""
                    yield {"_index": config.ES_INDEX_CAPTION, "_id": f"{v_id}_{int(r['frame_id'])}", "_source": {"video_id": v_id, "frame_id": int(r["frame_id"]), "text": str(text)}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES Caption Info] {e}")
        return False

def ensure_ocr_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_OCR): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_OCR, mappings={"properties": {"video_id": {"type": "keyword"}, "frame_id": {"type": "integer"}, "text": {"type": "text"}}})
        def _docs():
            if not config.OCR_DIR.exists(): return
            for csv_path in sorted(config.OCR_DIR.glob("*.csv")):
                if csv_path.name.startswith("run_manifest"): continue
                video_id = csv_path.stem
                df = pd.read_csv(csv_path)
                if df.empty or "frame_id" not in df.columns: continue
                text_col = "text" if "text" in df.columns else df.columns[-1]
                grouped = df.groupby("frame_id")[text_col].apply(lambda s: " ".join(str(t) for t in s if pd.notna(t)))
                for frame_id, text in grouped.items():
                    if not text.strip(): continue
                    yield {"_index": config.ES_INDEX_OCR, "_id": f"{video_id}_{int(frame_id)}", "_source": {"video_id": video_id, "frame_id": int(frame_id), "text": text}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES OCR Info] {e}")
        return False

def ensure_summary_fuzzy_index():
    try:
        from elasticsearch import helpers
        es = get_es_client()
        if not es.ping(): return False
        try:
            if es.indices.exists(index=config.ES_INDEX_SUMMARY): return True
        except Exception:
            return False
        es.indices.create(index=config.ES_INDEX_SUMMARY, mappings={"properties": {"video_id": {"type": "keyword"}, "text": {"type": "text"}}})
        def _docs():
            if not config.SUMMARY_DIR.exists(): return
            for txt_path in sorted(config.SUMMARY_DIR.glob("*.txt")):
                video_id = txt_path.stem
                text = txt_path.read_text(encoding="utf-8").strip()
                if not text: continue
                yield {"_index": config.ES_INDEX_SUMMARY, "_id": video_id, "_source": {"video_id": video_id, "text": text}}
        helpers.bulk(es, _docs(), stats_only=True, raise_on_error=False)
        return True
    except Exception as e:
        print(f"[ES Summary Info] {e}")
        return False

def ensure_all_fuzzy_indices():
    for fn in [ensure_asr_fuzzy_index, ensure_caption_fuzzy_index, ensure_ocr_fuzzy_index, ensure_summary_fuzzy_index]:
        try:
            fn()
        except Exception as e:
            print(f"[ES Info] {fn.__name__}: {e}")
'''
(WORKSPACE_DIR / "backend/es_indexing.py").write_text(es_indexing_code, encoding="utf-8")

# 6. Ghi đè trực tiếp keyframe.py chuẩn an toàn cao
keyframe_code = '''import glob as glob_mod
import faiss
import numpy as np
import pandas as pd
from cachetools import TTLCache

from .. import config
from ..common import l2_normalize, query_hash, video_id_from_filename
from ..models import is_image_query, siglip2_query_vec

try:
    import clip_encoder
except Exception:
    clip_encoder = None

_FRAME_INDICES: dict = {}

def build_frame_index(glob_pattern: str):
    npy_paths = sorted(glob_mod.glob(glob_pattern))
    if not npy_paths:
        print(f"[Keyframe Warning] No .npy files matched: {glob_pattern}. Initializing empty index.")
        dim = 768 if "siglip" in glob_pattern.lower() else 512
        index = faiss.IndexFlatIP(dim)
        result = (index, pd.DataFrame(columns=["video_id", "frame_id"]))
        _FRAME_INDICES[glob_pattern] = result
        return result

    all_vecs = []
    lookup_rows = []
    for npy_path in npy_paths:
        video_id = video_id_from_filename(npy_path, ("_viclip768", "_clip32", "_siglip768", "_siglip2"))
        vecs = np.load(npy_path).astype("float32")
        if vecs.ndim == 1:
            vecs = vecs.reshape(1, -1)
        for row_idx in range(len(vecs)):
            lookup_rows.append({"video_id": video_id, "frame_id": row_idx})
        all_vecs.append(vecs)

    matrix = l2_normalize(np.vstack(all_vecs).astype("float32"))
    index = faiss.IndexFlatIP(matrix.shape[1])
    index.add(matrix)
    result = (index, pd.DataFrame(lookup_rows))
    _FRAME_INDICES[glob_pattern] = result
    return result

def _get_frame_index(glob_pattern: str):
    if glob_pattern not in _FRAME_INDICES:
        build_frame_index(glob_pattern)
    return _FRAME_INDICES[glob_pattern]

def _search_frame(index, lookup_df, qvec: np.ndarray, k: int) -> pd.DataFrame:
    if index.ntotal == 0 or lookup_df.empty:
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    q = l2_normalize(qvec.reshape(1, -1))
    n = min(k, index.ntotal)
    scores, ids = index.search(q, n)
    valid_mask = (ids[0] >= 0) & (ids[0] < len(lookup_df))
    valid_ids = ids[0][valid_mask]
    if len(valid_ids) == 0:
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    results = lookup_df.iloc[valid_ids].copy().reset_index(drop=True)
    results["score"] = scores[0][valid_mask]
    results["rank"] = np.arange(1, len(results) + 1)
    results["n"] = results["frame_id"] + 1
    return results[["rank", "score", "video_id", "frame_id", "n"]]

_siglip2_cache = TTLCache(maxsize=256, ttl=300)
_clip_cache = TTLCache(maxsize=256, ttl=300)

def search_siglip2_frame(query, k: int = config.FETCH_K) -> pd.DataFrame:
    cache_key = (query_hash(query), k)
    if cache_key in _siglip2_cache:
        return _siglip2_cache[cache_key]
    index, lookup_df = _get_frame_index(config.FRAME_SIGLIP2_GLOB)
    qvec = siglip2_query_vec(query)
    result = _search_frame(index, lookup_df, qvec, k)
    _siglip2_cache[cache_key] = result
    return result

def search_clip_frame(query, k: int = config.FETCH_K) -> pd.DataFrame:
    if is_image_query(query) or clip_encoder is None:
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    cache_key = (query_hash(query), k)
    if cache_key in _clip_cache:
        return _clip_cache[cache_key]
    index, lookup_df = _get_frame_index(config.FRAME_CLIP_GLOB)
    if index.ntotal == 0 or lookup_df.empty:
        return pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    try:
        qvec = clip_encoder.encode_text([query])[0]
        result = _search_frame(index, lookup_df, qvec, k)
    except Exception:
        result = pd.DataFrame(columns=["rank", "score", "video_id", "frame_id", "n"])
    _clip_cache[cache_key] = result
    return result

def rrf_fuse_frame(dfs: list, k: int = config.RRF_K, top_n: int = config.DISPLAY_N) -> pd.DataFrame:
    scores, extra = {}, {}
    for df in dfs:
        if df is None or df.empty:
            continue
        for _, row in df.iterrows():
            key = (row["video_id"], int(row["frame_id"]))
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + row["rank"])
            extra.setdefault(key, {"n": int(row["frame_id"]) + 1})
    rows = [{"video_id": vid, "frame_id": fid, "rrf_score": s, **extra[(vid, fid)]}
            for (vid, fid), s in scores.items()]
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out = out.sort_values("rrf_score", ascending=False).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out.head(top_n)
'''
(WORKSPACE_DIR / "backend/search/keyframe.py").write_text(keyframe_code, encoding="utf-8")

# 7. Ghi đè trực tiếp backend/es_client.py dùng 127.0.0.1 và hỗ trợ force_new
es_client_code = '''from elasticsearch import Elasticsearch
from . import config

_client = None

def get_es_client(force_new: bool = False):
    global _client
    if _client is None or force_new:
        host = str(config.ES_HOST).replace("localhost", "127.0.0.1")
        if not host.startswith("http"):
            host = f"http://{host}"
        _client = Elasticsearch(
            hosts=[host],
            request_timeout=5,
            verify_certs=False,
            ssl_show_warn=False,
            meta_header=False,
        )
    return _client
'''
(WORKSPACE_DIR / "backend/es_client.py").write_text(es_client_code, encoding="utf-8")

# 8. Ghi đè trực tiếp backend/common.py & backend/routes/playback.py hỗ trợ HTTP 206 Streaming & Degarr video scan
(WORKSPACE_DIR / "backend/common.py").write_text('"""\nbackend/common.py -- shared helpers used by every search/* module. Ported\nfrom ui/app.py:170-276, 1315-1331 (video_id_from_filename through\nimage_b64, plus df_to_results). One behavioral change from the Streamlit\napp: thumbnails/video are served as URLs via FastAPI\'s StaticFiles mount\n(backend/main.py) instead of base64 data URIs, so `thumbnail_path` becomes\n`thumbnail_url` and there\'s no `image_b64` equivalent needed here -- see\nthe rewrite plan\'s Decisions section 1.\n"""\n\nimport hashlib\nimport re\nfrom pathlib import Path\n\nimport faiss\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nfrom . import config\n\n\ndef query_hash(query) -> str:\n    """Cache-key fragment for a query -- a text query is already hashable\n    as-is, but a picture query (PIL.Image) isn\'t, so hash its raw pixel\n    bytes instead (mirrors ui/app.py\'s _QUERY_HASH_FUNCS, which existed for\n    the same reason: st.cache_data\'s default hasher doesn\'t know how to\n    fingerprint a PIL.Image either)."""\n    if isinstance(query, Image.Image):\n        return "img:" + hashlib.sha1(query.tobytes()).hexdigest()\n    return "txt:" + query\n\n# ---------------------------------------------------------------------------\n# Filename / id parsing\n# ---------------------------------------------------------------------------\n\ndef video_id_from_filename(path_str: str, suffixes: tuple) -> str:\n    stem = Path(path_str).stem\n    for suffix in suffixes:\n        if stem.endswith(suffix):\n            return stem[: -len(suffix)]\n    return stem\n\n\ndef l2_normalize(mat: np.ndarray) -> np.ndarray:\n    mat = mat.astype("float32", copy=True)\n    faiss.normalize_L2(mat)\n    return mat\n\n\ndef parse_lot_range(text: str, exclude: bool = False):\n    """\'L21-L30\' / \'P01-P10\' / \'21-30\' / \'L21\' / \'P01\' -> (lo, hi, exclude) lot range, or None if\n    blank/unparsable. `exclude` flips apply_filters from "keep only this\n    range" (the default) to "drop this range" -- the sidebar\'s "Exclude"\n    checkbox next to "Search in collection"."""\n    text = (text or "").strip().upper()\n    if not text:\n        return None\n    m = re.match(r"^(?:L|P|ADL_?)?(\\d+)\\s*-\\s*(?:L|P|ADL_?)?(\\d+)$", text)\n    if m:\n        lo, hi = int(m.group(1)), int(m.group(2))\n        lo, hi = (lo, hi) if lo <= hi else (hi, lo)\n        return (lo, hi, exclude)\n    m = re.match(r"^(?:L|P|ADL_?)?(\\d+)$", text)\n    if m:\n        n = int(m.group(1))\n        return (n, n, exclude)\n    return None\n\n\ndef video_lot_num(video_id: str):\n    m = re.match(r"^(?:L|P|ADL_?)?(\\d+)", str(video_id).upper())\n    return int(m.group(1)) if m else None\n\n\ndef video_lot_str(video_id: str) -> str:\n    lot = video_lot_num(video_id)\n    if lot is not None:\n        prefix = "P" if str(video_id).upper().startswith("P") else "L"\n        return f"{prefix}{lot}"\n    return str(video_id)\n\n\ndef apply_filters(df: pd.DataFrame, video_filter: str, lot_range) -> pd.DataFrame:\n    """Restrict a leg\'s result df to a single video_id and/or a lot range\n    (or, when that range\'s exclude flag is set, drop it instead of keeping\n    only it), applied right after search (before RRF/head truncation) so\n    both single-leg and RRF views respect the same filters."""\n    if df is None or df.empty:\n        return df\n    out = df\n    video_filter = (video_filter or "").strip().upper()\n    if video_filter:\n        out = out[out["video_id"].astype(str).str.upper() == video_filter]\n    if lot_range:\n        lo, hi, exclude = lot_range\n        lots = out["video_id"].map(video_lot_num)\n        in_range = lots.notna() & (lots >= lo) & (lots <= hi)\n        out = out[~in_range] if exclude else out[in_range]\n    return out.reset_index(drop=True)\n\n\n# ---------------------------------------------------------------------------\n# Thumbnails / video / map-keyframes\n# ---------------------------------------------------------------------------\n\ndef thumbnail_url(video_id: str, n) -> str:\n    if n is None or pd.isna(n):\n        return ""\n    return f"/media/keyframes/{video_id}/{int(n):03d}.jpg"\n\n\ndef thumbnail_disk_path(video_id: str, n) -> Path:\n    return config.THUMBNAIL_ROOT / video_id / f"{int(n):03d}.jpg"\n\n\n_video_path_cache: dict = {}\n_all_video_files_index: dict = {}\n_has_indexed_kaggle_videos: bool = False\n\n\ndef _build_kaggle_video_index():\n    global _has_indexed_kaggle_videos\n    if _has_indexed_kaggle_videos:\n        return\n    _has_indexed_kaggle_videos = True\n    input_root = Path("/kaggle/input")\n    if not input_root.exists():\n        return\n    # Scan /kaggle/input for all video files (mp4, mkv, webm, avi)\n    exts = {".mp4", ".mkv", ".webm", ".avi"}\n    for root, _, files in os.walk(input_root):\n        for f in files:\n            p = Path(root) / f\n            if p.suffix.lower() in exts:\n                _all_video_files_index[p.stem.lower()] = p\n\n\ndef find_video_path(video_id: str) -> Path | None:\n    """Finds the actual video file path for a given video_id across flat directories,\n    nested lot directories (Videos_L23, L23, video/), or Kaggle input mounts (e.g. degarr)."""\n    if not video_id:\n        return None\n    vid_clean = str(video_id).strip()\n    vid_key = vid_clean.lower()\n    if vid_key in _video_path_cache:\n        return _video_path_cache[vid_key]\n\n    exts = [".mp4", ".mkv", ".webm", ".avi"]\n\n    # 1. Direct in config.VIDEO_DIR\n    if config.VIDEO_DIR and config.VIDEO_DIR.exists():\n        for ext in exts:\n            p = config.VIDEO_DIR / f"{vid_clean}{ext}"\n            if p.exists():\n                _video_path_cache[vid_key] = p\n                return p\n\n        # Check lot subfolders within config.VIDEO_DIR\n        lot_str = video_lot_str(vid_clean)\n        candidate_subdirs = [\n            config.VIDEO_DIR / lot_str,\n            config.VIDEO_DIR / f"Videos_{lot_str}",\n            config.VIDEO_DIR / f"Videos_{lot_str}" / "video",\n            config.VIDEO_DIR / "video" / lot_str,\n            config.VIDEO_DIR / "video",\n            config.VIDEO_DIR / "videos",\n        ]\n        for sdir in candidate_subdirs:\n            if sdir.exists():\n                for ext in exts:\n                    p = sdir / f"{vid_clean}{ext}"\n                    if p.exists():\n                        _video_path_cache[vid_key] = p\n                        return p\n\n    # 2. Check Kaggle /kaggle/input (datasets like degarr, aic2026-dataset, etc.)\n    if config.IS_KAGGLE or Path("/kaggle/input").exists():\n        _build_kaggle_video_index()\n        if vid_key in _all_video_files_index and _all_video_files_index[vid_key].exists():\n            res = _all_video_files_index[vid_key]\n            _video_path_cache[vid_key] = res\n            return res\n\n    _video_path_cache[vid_key] = None\n    return None\n\n\ndef video_url(video_id: str) -> str:\n    return f"/api/playback/stream/{video_id}"\n\n\n_map_keyframes_cache: dict = {}\n\n\ndef load_map_keyframes(video_id: str):\n    if video_id not in _map_keyframes_cache:\n        path = config.MAP_KEYFRAMES_DIR / f"{video_id}.csv"\n        if path.exists():\n            df = pd.read_csv(path)\n            # Defends against stray junk on the header row (seen on one file\n            # in the wild: a leading backtick turned "n" into "`n", which\n            # made every mk["n"] lookup below raise KeyError and 500 the\n            # whole request) -- normalize instead of trusting the header\n            # byte-for-byte, so a similarly mangled file degrades to working\n            # rather than crashing.\n            df.columns = [str(c).strip().strip("`") for c in df.columns]\n            _map_keyframes_cache[video_id] = df\n        else:\n            _map_keyframes_cache[video_id] = None\n    return _map_keyframes_cache[video_id]\n\n\ndef get_video_keyframes_meta(video_id: str) -> list:\n    """Returns a list of keyframe descriptors for video_id for frontend scrubber/fallback."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty:\n        return []\n    out = []\n    for _, r in mk.iterrows():\n        try:\n            n_val = int(r["n"])\n            pts_val = float(r.get("pts_time", 0.0))\n            fps_val = float(r.get("fps", 25.0))\n            frame_idx_val = int(r.get("frame_idx", n_val))\n            out.append({\n                "n": n_val,\n                "pts_time": pts_val,\n                "fps": fps_val,\n                "frame_idx": frame_idx_val,\n                "thumbnail_url": thumbnail_url(video_id, n_val),\n            })\n        except Exception:\n            continue\n    return out\n\n\ndef nearest_keyframe_n_by_time(video_id: str, t: float):\n    """Nearest map-keyframes row (by pts_time) to timestamp t -- used for\n    the ASR fuzzy leg, which is segment-level only (no direct frame_id)."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty or pd.isna(t):\n        return None\n    idx = (mk["pts_time"] - t).abs().idxmin()\n    return int(mk.loc[idx, "n"])\n\n\ndef keyframe_timestamp(video_id: str, n):\n    """Symmetric counterpart to nearest_keyframe_n_by_time: direct n ->\n    (pts_time, fps) lookup, used by TRAKE to place a matched frame on the\n    video timeline. Returns (None, None) if unresolvable."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty or n is None or pd.isna(n):\n        return None, None\n    hit = mk.loc[mk["n"] == int(n)]\n    if hit.empty:\n        return None, None\n    row = hit.iloc[0]\n    return float(row["pts_time"]), float(row["fps"])\n\n\ndef frame_idx_for_n(video_id: str, n):\n    """n (1-indexed keyframe ordinal, the field every result carries) ->\n    frame_idx (raw video frame index) -- used by backend/export.py, since\n    AIC submission CSVs expect frame_idx, not n. Returns None if\n    unresolvable."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty or n is None or pd.isna(n):\n        return None\n    hit = mk.loc[mk["n"] == int(n)]\n    if hit.empty:\n        return None\n    return int(hit.iloc[0]["frame_idx"])\n\n\ndef valid_ns_for_video(video_id: str) -> set:\n    """Every n that actually exists for video_id -- used by backend/export.py\n    to filter out-of-range hedge offsets (n-1, n+2, ...) before they\'d\n    resolve to a nonexistent keyframe."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty:\n        return set()\n    return set(mk["n"].astype(int))\n\n\ndef n_for_frame_idx(video_id: str, frame_idx: int):\n    """Reverse of frame_idx_for_n: native frame_idx -> keyframe n, only if\n    frame_idx exactly matches an existing keyframe\'s frame_idx. Used by\n    TRAKE row generation (backend/export.py::generate_trake_rows) to tell\n    apart a keyframe-backed pick (neighbours ranked by keyframe-index\n    distance) from a raw native-frame pick with no embedding behind it\n    (neighbours ranked by plain frame-number distance instead). Returns\n    None if frame_idx isn\'t any keyframe\'s, including when map-keyframes\n    itself is missing for video_id."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty or frame_idx is None:\n        return None\n    hit = mk.loc[mk["frame_idx"] == int(frame_idx)]\n    if hit.empty:\n        return None\n    return int(hit.iloc[0]["n"])\n\n\ndef native_frame_range_for_video(video_id: str):\n    """(lo, hi) bound on real frame_idx values for video_id, used by TRAKE\n    row generation to keep interpolated/hedge frame numbers in range. Only\n    as precise as map-keyframes\' own frame_idx column (the true last video\n    frame can run a little past the last keyframe\'s) -- good enough for a\n    filler-row bound, not claimed to be the exact video length. Falls back\n    to a generous synthetic range if map-keyframes is missing entirely, so\n    callers never have to special-case "no data" themselves."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty:\n        return (0, 10**7)\n    return (0, int(mk["frame_idx"].max()))\n\n\ndef video_fps_for_video(video_id: str) -> float:\n    """Any one row\'s fps for video_id -- map-keyframes stores the same fps\n    on every row for a given video. Used to start TRAKE curation playback\n    from a bare video_id, before any keyframe/timestamp is known yet.\n    Falls back to the same 25.0 default backend/routes/playback.py already\n    uses when a specific frame\'s fps can\'t be resolved."""\n    mk = load_map_keyframes(video_id)\n    if mk is None or mk.empty:\n        return 25.0\n    return float(mk.iloc[0]["fps"])\n\n\n# ---------------------------------------------------------------------------\n# Result-shape contract -- every leg/signal normalizes to this dict shape\n# before it\'s returned to the frontend (mirrors ui/app.py\'s df_to_results).\n# ---------------------------------------------------------------------------\n\ndef df_to_results(df: pd.DataFrame, score_col: str, text_col: str = None) -> list:\n    if df is None or df.empty:\n        return []\n    out = []\n    for _, r in df.iterrows():\n        n = r.get("n")\n        if n is None or pd.isna(n):\n            continue\n        n = int(n)\n        text = r.get(text_col) if text_col else None\n        out.append({\n            "video_id": r["video_id"], "n": n, "rank": int(r["rank"]),\n            "score_label": score_col, "score_val": float(r[score_col]),\n            "text": text if isinstance(text, str) else None,\n            "thumbnail_url": thumbnail_url(r["video_id"], n),\n        })\n    return out\n', encoding="utf-8")
(WORKSPACE_DIR / "backend/routes/playback.py").write_text('"""\nbackend/routes/playback.py -- video streaming and playback metadata endpoint.\nSupports:\n- HTTP 206 Partial Content (Byte Range Requests) for HTML5 <video> scrubbing.\n- Smart auto-discovery of video files across lot folders and Kaggle dataset inputs (e.g. degarr).\n- Rich keyframe metadata and fallback descriptors when raw video is missing.\n"""\n\nimport os\nfrom typing import Optional\nfrom pathlib import Path\nimport mimetypes\n\nfrom fastapi import APIRouter, Header, HTTPException, Request\nfrom fastapi.responses import StreamingResponse, FileResponse, Response\n\nfrom .. import config\nfrom ..common import (\n    find_video_path,\n    get_video_keyframes_meta,\n    keyframe_timestamp,\n    video_fps_for_video,\n    video_url,\n)\n\nrouter = APIRouter()\n\n\n@router.get("/api/playback")\ndef get_playback(video_id: str, n: Optional[int] = None):\n    vpath = find_video_path(video_id)\n    has_video = vpath is not None and vpath.exists()\n    \n    if n is None:\n        ts, fps = 0, video_fps_for_video(video_id)\n    else:\n        ts, fps = keyframe_timestamp(video_id, n)\n\n    keyframes = get_video_keyframes_meta(video_id)\n    \n    return {\n        "video_id": video_id,\n        "has_video": has_video,\n        "video_url": video_url(video_id),\n        "start_time": ts if ts is not None else 0,\n        "fps": fps if fps is not None else 25.0,\n        "total_keyframes": len(keyframes),\n        "keyframes": keyframes,\n    }\n\n\ndef _file_chunk_generator(file_path: Path, start: int, chunk_size: int, block_size: int = 1024 * 1024):\n    with open(file_path, "rb") as f:\n        f.seek(start)\n        remaining = chunk_size\n        while remaining > 0:\n            bytes_to_read = min(remaining, block_size)\n            data = f.read(bytes_to_read)\n            if not data:\n                break\n            remaining -= len(data)\n            yield data\n\n\n@router.get("/api/playback/stream/{video_id}")\nasync def stream_video(video_id: str, request: Request, range: Optional[str] = Header(None)):\n    vpath = find_video_path(video_id)\n    if not vpath or not vpath.exists():\n        raise HTTPException(404, f"Video stream not found for {video_id}.")\n\n    file_size = vpath.stat().st_size\n    mime_type, _ = mimetypes.guess_type(str(vpath))\n    content_type = mime_type or "video/mp4"\n\n    # Handle Byte-Range Request (HTTP 206)\n    if range:\n        try:\n            # Parse \'bytes=start-end\'\n            range_str = range.strip()\n            if range_str.startswith("bytes="):\n                range_str = range_str[6:]\n            parts = range_str.split("-")\n            start = int(parts[0]) if parts[0] else 0\n            end = int(parts[1]) if len(parts) > 1 and parts[1] else file_size - 1\n            start = max(0, min(start, file_size - 1))\n            end = max(start, min(end, file_size - 1))\n            chunk_size = (end - start) + 1\n\n            headers = {\n                "Content-Range": f"bytes {start}-{end}/{file_size}",\n                "Accept-Ranges": "bytes",\n                "Content-Length": str(chunk_size),\n                "Content-Type": content_type,\n                "Access-Control-Allow-Origin": "*",\n                "Access-Control-Allow-Headers": "Range, Accept-Ranges, Content-Range",\n            }\n            return StreamingResponse(\n                _file_chunk_generator(vpath, start, chunk_size),\n                status_code=206,\n                headers=headers,\n            )\n        except Exception:\n            pass\n\n    # Full content response with Accept-Ranges support\n    headers = {\n        "Accept-Ranges": "bytes",\n        "Content-Length": str(file_size),\n        "Content-Type": content_type,\n        "Access-Control-Allow-Origin": "*",\n        "Access-Control-Allow-Headers": "Range, Accept-Ranges, Content-Range",\n    }\n    return StreamingResponse(\n        _file_chunk_generator(vpath, 0, file_size),\n        status_code=200,\n        headers=headers,\n    )\n', encoding="utf-8")

# 9. Ghi đè trực tiếp frontend UI: dialogs.js, export-ui.js, style.css
(WORKSPACE_DIR / "frontend/js/dialogs.js").write_text('// frontend/js/dialogs.js -- modal dialogs. Phase 1: "Nearby frames" (ports\n// ui/app.py\'s show_neighbors, ui/app.py:1334-1360) and single-frame\n// playback (ports frame_playback_dialog, ui/app.py:1363-1376). TRAKE\'s\n// marker-bar playback and the Mixed change-weights dialog land in their\n// own phases, same file.\n\nimport { getNeighbors, getPlayback } from "./api.js";\n// export-dialog.js now just opens the Export CSV tab (frontend/export.html,\n// see its own header) rather than a modal built from openDialog() here --\n// no circular import with this file any more.\nimport { openExportDialog } from "./export-dialog.js";\nimport {\n    getNeighborExtra, MIXED_DEFAULT_LEGS, MIXED_DEFAULT_WEIGHTS,\n    MIXED_LEG_DEFS, MIXED_SIGNAL_NAMES, mixedConfig, saveMixedConfig,\n} from "./state.js";\n\nconst root = document.getElementById("dialog-root");\n\n// `title` may be falsy (null/"") to omit the heading entirely -- used by\n// the playback dialogs, which put video_id/frame/timer info beside the\n// video instead of needing a heading above it. The close button is\n// absolutely positioned (not floated) specifically so it works the same\n// way whether or not a title/h3 is present.\nexport function openDialog(title, bodyEl, { wide = false } = {}) {\n    const overlay = document.createElement("div");\n    overlay.className = "dialog-overlay";\n    const box = document.createElement("div");\n    box.className = "dialog-box" + (wide ? " wide" : "");\n    const closeBtn = document.createElement("button");\n    closeBtn.className = "dialog-close";\n    closeBtn.textContent = "✕";\n    closeBtn.onclick = () => overlay.remove();\n    overlay.onclick = (e) => { if (e.target === overlay) overlay.remove(); };\n\n    box.append(closeBtn);\n    if (title) {\n        const h3 = document.createElement("h3");\n        h3.textContent = title;\n        box.append(h3);\n    }\n    box.append(bodyEl);\n    overlay.append(box);\n    root.append(overlay);\n    return { overlay, box };\n}\n\nexport async function openNeighborsDialog(videoId, centerN) {\n    const body = document.createElement("div");\n    body.innerHTML = `<div class="thumb-caption" style="margin-bottom:0.5rem;">${videoId} — around frame ${centerN}</div>\n        <button class="btn" id="nbr-up" style="width:100%;margin-bottom:0.5rem;">▲ 10 earlier</button>\n        <div class="grid nbr-grid" id="nbr-grid" style="grid-template-columns:repeat(5,1fr);"></div>\n        <button class="btn" id="nbr-down" style="width:100%;margin-top:0.5rem;">▼ 10 later</button>`;\n    const { box } = openDialog("Nearby frames", body, { wide: true });\n\n    async function refresh() {\n        const extra = getNeighborExtra(videoId, centerN);\n        const data = await getNeighbors(videoId, centerN, extra.before, extra.after);\n        const grid = box.querySelector("#nbr-grid");\n        grid.innerHTML = "";\n        for (const f of data.frames) {\n            const cell = document.createElement("div");\n            cell.className = "thumb-cell";\n            // thumb-wrap-static suppresses the normal hover-zoom (same\n            // class TRAKE\'s low-count cards use) -- distracting in this\n            // tightly packed nearby-frames grid.\n            cell.innerHTML = f.exists\n                ? `<div class="thumb-wrap thumb-wrap-static"><img src="${f.thumbnail_url}"></div>`\n                : `<div class="thumb-missing">(missing)</div>`;\n            const cap = document.createElement("div");\n            cap.className = "thumb-caption";\n            cap.innerHTML = f.is_center ? `<b>${f.n}</b>` : String(f.n);\n            cell.append(cap);\n            if (f.exists) {\n                // Reuses .export-add-btn\'s CSS (top-right overlay corner,\n                // same as the export screen\'s own preview cards) purely for\n                // position/sizing -- unrelated to that button\'s add/replace\n                // behavior elsewhere.\n                const exportBtn = document.createElement("button");\n                exportBtn.className = "icon-btn export-add-btn";\n                exportBtn.title = "Export as AIC submission CSV";\n                exportBtn.textContent = "★";\n                exportBtn.onclick = () => openExportDialog({ kind: "flat", video_id: videoId, n: f.n });\n                cell.append(exportBtn);\n            }\n            grid.append(cell);\n        }\n    }\n\n    box.querySelector("#nbr-up").onclick = () => {\n        getNeighborExtra(videoId, centerN).before += 10;\n        refresh();\n    };\n    box.querySelector("#nbr-down").onclick = () => {\n        getNeighborExtra(videoId, centerN).after += 10;\n        refresh();\n    };\n\n    await refresh();\n}\n\nfunction fmtTime(t) {\n    const mm = String(Math.floor(t / 60)).padStart(2, "0");\n    const ss = (t % 60).toFixed(2).padStart(5, "0");\n    return `${mm}:${ss}`;\n}\n\nfunction renderKeyframePlayer(wrap, timerEl, videoId, data, initialN, onExportReady) {\n    const keyframes = data.keyframes || [];\n    let currentIdx = keyframes.findIndex((k) => k.n === Number(initialN));\n    if (currentIdx === -1) currentIdx = 0;\n\n    let isPlaying = false;\n    let playInterval = null;\n\n    wrap.innerHTML = `\n      <div class="keyframe-player">\n        <div class="keyframe-player-img-wrap">\n          <img class="keyframe-player-img" id="kf-img" src="" alt="Keyframe Preview">\n        </div>\n        <div class="keyframe-controls-bar">\n          <button class="keyframe-ctrl-btn" id="kf-prev" title="Previous keyframe">◀</button>\n          <button class="keyframe-ctrl-btn" id="kf-play" title="Play / Pause Slideshow">▶</button>\n          <button class="keyframe-ctrl-btn" id="kf-next" title="Next keyframe">▶</button>\n          <input type="range" class="keyframe-slider" id="kf-slider" min="0" max="${Math.max(0, keyframes.length - 1)}" value="${currentIdx}">\n          <span id="kf-counter" style="font-size:0.8rem;color:#cbd5e1;white-space:nowrap;">${currentIdx + 1} / ${keyframes.length}</span>\n        </div>\n      </div>\n    `;\n\n    const img = wrap.querySelector("#kf-img");\n    const slider = wrap.querySelector("#kf-slider");\n    const counter = wrap.querySelector("#kf-counter");\n    const playBtn = wrap.querySelector("#kf-play");\n\n    function showKeyframe(idx) {\n        if (!keyframes.length) return;\n        currentIdx = Math.max(0, Math.min(idx, keyframes.length - 1));\n        const kf = keyframes[currentIdx];\n        img.src = kf.thumbnail_url;\n        slider.value = currentIdx;\n        counter.textContent = `${currentIdx + 1} / ${keyframes.length}`;\n        timerEl.textContent = `${fmtTime(kf.pts_time)} · frame ${kf.frame_idx || Math.round(kf.pts_time * kf.fps)}`;\n        if (onExportReady) {\n            onExportReady({ n: kf.n, frame_idx: kf.frame_idx || Math.round(kf.pts_time * kf.fps) });\n        }\n    }\n\n    wrap.querySelector("#kf-prev").onclick = () => {\n        stopSlideshow();\n        showKeyframe(currentIdx - 1);\n    };\n    wrap.querySelector("#kf-next").onclick = () => {\n        stopSlideshow();\n        showKeyframe(currentIdx + 1);\n    };\n    slider.oninput = () => {\n        stopSlideshow();\n        showKeyframe(parseInt(slider.value, 10));\n    };\n\n    function startSlideshow() {\n        isPlaying = true;\n        playBtn.textContent = "⏸";\n        playInterval = setInterval(() => {\n            if (currentIdx >= keyframes.length - 1) {\n                showKeyframe(0);\n            } else {\n                showKeyframe(currentIdx + 1);\n            }\n        }, 350);\n    }\n\n    function stopSlideshow() {\n        if (isPlaying) {\n            isPlaying = false;\n            playBtn.textContent = "▶";\n            clearInterval(playInterval);\n            playInterval = null;\n        }\n    }\n\n    playBtn.onclick = () => {\n        if (isPlaying) stopSlideshow();\n        else startSlideshow();\n    };\n\n    showKeyframe(currentIdx);\n}\n\nexport async function openPlaybackDialog(videoId, n) {\n    const body = document.createElement("div");\n    body.innerHTML = `<div class="playback-layout">\n        <div class="playback-main" id="playback-video-wrap">\n          <div class="playback-container">\n            <div class="playback-loading">⏳ Loading video playback…</div>\n          </div>\n        </div>\n        <div class="playback-info">\n          <div class="thumb-caption"><b>${videoId}</b> — frame ${n || 1}</div>\n          <div id="playback-timer" class="playback-timer">--:-- · frame --</div>\n          <div id="playback-mode-banner" style="margin-top:0.4rem;font-size:0.75rem;color:var(--text-muted);"></div>\n          <button class="btn" id="playback-export-btn" style="margin-top:0.5rem;width:100%;" title="Export the exact frame currently playing">★ Export this frame</button>\n        </div>\n      </div>`;\n    const { box } = openDialog(null, body, { wide: true });\n\n    let currentExportPayload = { kind: "flat", video_id: videoId, n: n };\n    const timer = box.querySelector("#playback-timer");\n    const wrap = box.querySelector("#playback-video-wrap");\n    const modeBanner = box.querySelector("#playback-mode-banner");\n\n    box.querySelector("#playback-export-btn").onclick = () => {\n        openExportDialog(currentExportPayload);\n    };\n\n    try {\n        const data = await getPlayback(videoId, n);\n        wrap.innerHTML = "";\n\n        if (data.has_video) {\n            modeBanner.textContent = "🎥 Video playback mode";\n            const video = document.createElement("video");\n            video.src = data.video_url;\n            video.controls = true;\n            video.autoplay = false;\n            video.preload = "metadata";\n            video.playsInline = true;\n\n            video.addEventListener("loadedmetadata", () => {\n                if (data.start_time && isFinite(data.start_time) && data.start_time > 0) {\n                    video.currentTime = data.start_time;\n                }\n                timer.textContent = `${fmtTime(video.currentTime)} · frame ${Math.round(video.currentTime * data.fps)}`;\n            });\n\n            video.addEventListener("timeupdate", () => {\n                const curFrame = Math.round(video.currentTime * data.fps);\n                timer.textContent = `${fmtTime(video.currentTime)} · frame ${curFrame}`;\n                currentExportPayload = { kind: "frame", video_id: videoId, frame_idx: curFrame };\n            });\n\n            video.onerror = () => {\n                // If MP4 streaming fails, gracefully fallback to keyframe scrubber\n                modeBanner.innerHTML = `<span style="color:#e67e22;">⚠️ Video stream unavailable. Keyframe preview active.</span>`;\n                wrap.innerHTML = "";\n                renderKeyframePlayer(wrap, timer, videoId, data, n, (kf) => {\n                    currentExportPayload = { kind: "flat", video_id: videoId, n: kf.n, frame_idx: kf.frame_idx };\n                });\n            };\n\n            const container = document.createElement("div");\n            container.className = "playback-container";\n            container.append(video);\n            wrap.append(container);\n        } else {\n            // No video file on disk: immediate fallback to Keyframe Timeline Scrubber\n            modeBanner.innerHTML = `<span style="color:#3b82f6;">🖼️ Keyframe timeline preview (MP4 not on disk)</span>`;\n            renderKeyframePlayer(wrap, timer, videoId, data, n, (kf) => {\n                currentExportPayload = { kind: "flat", video_id: videoId, n: kf.n, frame_idx: kf.frame_idx };\n            });\n        }\n    } catch (e) {\n        wrap.innerHTML = `<div class="status-banner error">${e.message}</div>`;\n    }\n}\n\n// "Change weights" dialog -- ports ui/app.py\'s change_weights_dialog\n// (ui/app.py:1286-1312). Edits a staged copy so Cancel discards changes;\n// Save commits into the one shared mixedConfig (state.js) and persists it,\n// same as ui/app.py committing into st.session_state.mixed_weights/legs.\n// `onSave` lets the caller (standalone Mixed mode, or a TRAKE row in a\n// later phase) re-run its search after a Save.\nexport function openWeightsDialog(onSave) {\n    const staged = {\n        weights: { ...mixedConfig.weights },\n        legs: { ...mixedConfig.legs },\n    };\n\n    const body = document.createElement("div");\n    body.innerHTML = `<div class="thumb-caption muted" style="margin-bottom:0.5rem;">Weight per signal (0 = off) with that signal\'s legs alongside</div>\n        <div id="weights-rows"></div>\n        <hr class="divider">\n        <div style="display:flex;gap:0.5rem;">\n          <button class="btn" id="weights-default">Default</button>\n          <button class="btn" id="weights-cancel">Cancel</button>\n          <button class="btn btn-primary" id="weights-save">Save</button>\n        </div>`;\n    const { overlay, box } = openDialog("Change weights", body);\n    const rows = box.querySelector("#weights-rows");\n\n    function renderRows() {\n        rows.innerHTML = "";\n        for (const name of MIXED_SIGNAL_NAMES) {\n            const row = document.createElement("div");\n            row.style.cssText = "display:flex;align-items:center;gap:1rem;margin:0.5rem 0;";\n            const label = document.createElement("label");\n            label.style.cssText = "flex:0 0 90px;margin:0;";\n            label.textContent = `${name} (${staged.weights[name]})`;\n            const slider = document.createElement("input");\n            slider.type = "range";\n            slider.min = "0"; slider.max = "3"; slider.step = "1";\n            slider.value = staged.weights[name];\n            slider.style.flex = "1 1 auto";\n            slider.oninput = () => {\n                staged.weights[name] = parseInt(slider.value, 10);\n                label.textContent = `${name} (${staged.weights[name]})`;\n            };\n            row.append(label, slider);\n\n            const legsBox = document.createElement("div");\n            legsBox.style.cssText = "flex:0 0 200px;display:flex;flex-direction:column;gap:0.15rem;";\n            if (MIXED_LEG_DEFS[name]) {\n                for (const [legKey, legLabel] of MIXED_LEG_DEFS[name]) {\n                    const cbRow = document.createElement("label");\n                    cbRow.style.cssText = "display:flex;align-items:center;gap:0.3rem;font-size:0.85rem;margin:0;";\n                    const cb = document.createElement("input");\n                    cb.type = "checkbox";\n                    cb.checked = staged.legs[legKey];\n                    cb.onchange = () => { staged.legs[legKey] = cb.checked; };\n                    cbRow.append(cb, document.createTextNode(legLabel));\n                    legsBox.append(cbRow);\n                }\n            } else {\n                legsBox.innerHTML = `<span class="thumb-caption muted">Detailed legs</span>`;\n            }\n            row.append(legsBox);\n            rows.append(row);\n        }\n    }\n    renderRows();\n\n    box.querySelector("#weights-default").onclick = () => {\n        staged.weights = { ...MIXED_DEFAULT_WEIGHTS };\n        staged.legs = { ...MIXED_DEFAULT_LEGS };\n        renderRows();\n    };\n    box.querySelector("#weights-cancel").onclick = () => overlay.remove();\n    box.querySelector("#weights-save").onclick = () => {\n        mixedConfig.weights = staged.weights;\n        mixedConfig.legs = staged.legs;\n        saveMixedConfig();\n        overlay.remove();\n        if (onSave) onSave();\n    };\n}\n\n// TRAKE\'s play-icon action: opens the source video seeked near the first\n// matched event, with a click-to-seek marker row for every matched event\n// and a live timestamp/frame readout. Ports trake_playback_dialog +\n// render_trake_playback_binder\'s JS (ui/app.py:1379-1492) -- collapsed\n// into one function here since a hand-written page has no\n// "st.dialog can\'t run <script>" limitation to work around, and no\n// rerun-driven observer/singleton-guard dance needed either.\nexport async function openTrakePlaybackDialog(videoId, events) {\n    const matched = events.filter((e) => e.matched && e.timestamp !== null);\n    const gaps = events.filter((e) => !e.matched);\n\n    const body = document.createElement("div");\n    body.innerHTML = `<div class="playback-layout">\n        <div class="playback-main">\n          <div id="trake-video-wrap">Loading…</div>\n          <div id="trake-marker-bar"></div>\n        </div>\n        <div class="playback-info">\n          <div class="thumb-caption">${videoId}</div>\n          <div id="trake-timer" class="playback-timer">--:-- · frame --</div>\n          <button class="btn" id="trake-export-btn" style="margin-top:0.5rem;" title="Export the exact frame currently playing">★ Export this frame</button>\n          <div id="trake-gaps"></div>\n        </div>\n      </div>`;\n    const { box } = openDialog(null, body, { wide: true });\n\n    if (gaps.length) {\n        const gapsEl = box.querySelector("#trake-gaps");\n        gapsEl.innerHTML = `<hr class="divider"><div class="thumb-caption muted" style="margin-bottom:0.4rem;">Coverage gaps — scrub manually between the nearest matched anchors:</div>`;\n        for (const e of gaps) {\n            const before = matched.filter((m) => m.event_index < e.event_index).at(-1);\n            const after = matched.find((m) => m.event_index > e.event_index);\n            const lo = before ? `${before.timestamp.toFixed(2)}s (${before.label})` : "start";\n            const hi = after ? `${after.timestamp.toFixed(2)}s (${after.label})` : "end";\n            const line = document.createElement("div");\n            line.className = "thumb-caption";\n            line.innerHTML = `${e.label}: no direct match — between <b>${lo}</b> and <b>${hi}</b>`;\n            gapsEl.append(line);\n        }\n    }\n\n    if (!matched.length) {\n        box.querySelector("#trake-video-wrap").innerHTML = `<div class="status-banner info">No matched events to seek to.</div>`;\n        const exportBtn = box.querySelector("#trake-export-btn");\n        exportBtn.disabled = true;\n        exportBtn.title = "No video loaded to read a frame from";\n        return;\n    }\n\n    try {\n        const data = await getPlayback(videoId, matched[0].n);\n        const wrap = box.querySelector("#trake-video-wrap");\n        const bar = box.querySelector("#trake-marker-bar");\n        const timer = box.querySelector("#trake-timer");\n        wrap.innerHTML = "";\n\n        let currentExportPayload = { kind: "frame", video_id: videoId, frame_idx: Math.round(matched[0].timestamp * data.fps) };\n        box.querySelector("#trake-export-btn").onclick = () => {\n            openExportDialog(currentExportPayload);\n        };\n\n        if (data.has_video) {\n            const video = document.createElement("video");\n            video.src = data.video_url;\n            video.controls = true;\n            video.preload = "metadata";\n            video.playsInline = true;\n\n            function layoutMarkers() {\n                if (!video.duration || !isFinite(video.duration)) return;\n                bar.innerHTML = "";\n                for (const m of matched) {\n                    const pct = Math.max(0, Math.min(100, (m.timestamp / video.duration) * 100));\n                    const tick = document.createElement("div");\n                    tick.className = "trake-marker-tick";\n                    tick.title = `${m.label} @ ${m.timestamp.toFixed(2)}s`;\n                    tick.textContent = m.label;\n                    tick.style.left = pct + "%";\n                    tick.addEventListener("click", () => { video.currentTime = m.timestamp; });\n                    bar.append(tick);\n                }\n            }\n\n            video.addEventListener("loadedmetadata", () => {\n                if (matched[0].timestamp && isFinite(matched[0].timestamp)) {\n                    video.currentTime = matched[0].timestamp;\n                }\n                layoutMarkers();\n            });\n            if (video.readyState >= 1) layoutMarkers();\n\n            video.addEventListener("timeupdate", () => {\n                const curFrame = Math.round(video.currentTime * data.fps);\n                timer.textContent = `${fmtTime(video.currentTime)} · frame ${curFrame}`;\n                currentExportPayload = { kind: "frame", video_id: videoId, frame_idx: curFrame };\n            });\n\n            video.onerror = () => {\n                bar.style.display = "none";\n                wrap.innerHTML = "";\n                renderKeyframePlayer(wrap, timer, videoId, data, matched[0].n, (kf) => {\n                    currentExportPayload = { kind: "frame", video_id: videoId, frame_idx: kf.frame_idx };\n                });\n            };\n\n            const container = document.createElement("div");\n            container.className = "playback-container";\n            container.append(video);\n            wrap.append(container);\n        } else {\n            bar.style.display = "none";\n            renderKeyframePlayer(wrap, timer, videoId, data, matched[0].n, (kf) => {\n                currentExportPayload = { kind: "frame", video_id: videoId, frame_idx: kf.frame_idx };\n            });\n        }\n    } catch (e) {\n        box.querySelector("#trake-video-wrap").innerHTML = `<div class="status-banner error">${e.message}</div>`;\n    }\n}\n', encoding="utf-8")
(WORKSPACE_DIR / "frontend/js/export-ui.js").write_text('// frontend/js/export-ui.js -- the actual Export CSV UI: query-type\n// segmented control, confirmed/unconfirmed answer curation, Neighbours/\n// Similars preview grids (KIS/VQA only), TRAKE\'s per-video event curation.\n// Mounted by export-page.js into the standalone Export CSV tab (frontend/export.html)\n// -- this module knows nothing about being in a separate tab: the host\n// supplies a plain container element, a `getCandidates` accessor (so\n// "Similars" can read a *different* tab\'s live search results without this\n// module importing that tab\'s state.js), and an `onDone()` callback for\n// "the user cancelled" (a completed export leaves the form in place instead\n// -- see #exp-export\'s handler -- so onDone is never called for that case).\n//\n// Two bodies depending on query type:\n//   KIS/VQA -- one frame\'s worth of query-answer curation (confirmed/\n//     unconfirmed) plus a two-section preview (nearest-by-time\n//     "Neighbours", already-ranked "Similars"). POSTs to /api/export and\n//     triggers the CSV download directly.\n//   TRAKE -- no confirmed/unconfirmed distinction at all any more. A\n//     curate -> cache -> merge flow instead:\n//       1. Curate one video at a time: an inline <video> preview (loaded\n//          by typing a video id, or seeded from `trigger`) plus an "Add"\n//          button that captures whatever frame is currently playing into\n//          an ordered event list (drag to reorder, ✕ to remove). No\n//          Neighbours/Similars preview for TRAKE -- the Frame ID box is\n//          the only other way to add an event, besides the video itself.\n//       2. "Generate rows" POSTs that video\'s {video_id, frame_idxs} to\n//          /api/export/trake-rows and caches the <=100 returned candidate\n//          sequences client-side, keyed by video_id (`s.trake.cache`\n//          below) -- repeatable for as many candidate videos as the human\n//          wants to compare, each just adding another entry.\n//       3. Export: the human checks which cached videos to include and\n//          their priority order; the rows are interleaved client-side (no\n//          backend round-trip, no re-reading anything) into one <=100-row\n//          set and POSTed to /api/export/trake-write, which only formats\n//          + returns CSV text for already-resolved rows -- the one file\n//          this whole flow ever writes to disk.\n//     All of one video\'s events still share that one video_id (the AIC\n//     TRAKE row format is one video per row) -- switching the curation\n//     panel to a different video starts a fresh event list, independent\n//     per-video cache entries are what let several candidate videos\n//     coexist for the merge step.\n//\n// `trigger` shapes:\n//   {kind: "flat", video_id, n}         -- any non-TRAKE signal\'s result card\n//   {kind: "trake", candidate}          -- a TRAKE candidate card\n//   {kind: "frame", video_id, frame_idx} -- a raw native frame from a video\n//     playback dialog, no keyframe n at all (TRAKE-only: KIS/VQA need an n\n//     for the backend\'s flat CSV path, which this trigger doesn\'t have).\n\nimport { exportCsv, getExportFrame, getExportNeighbors, getPlayback, getTrakeRows, writeTrakeCsv } from "./api.js";\n\nconst NEIGHBOUR_COUNT_EXPORT = 10; // fixed row-generation window, independent of preview expand state\nconst PREVIEW_PAGE = 12; // 3x4 grid per preview section (export-preview-grid is 4 columns wide)\n\nfunction freshState(trigger) {\n    const isFlat = trigger.kind === "flat";\n    const seed = isFlat ? { video_id: trigger.video_id, n: trigger.n } : null;\n\n    return {\n        trigger,\n        queryType: isFlat ? "KIS" : "TRAKE", // "trake"/"frame" triggers default to TRAKE -- "frame" has no n for KIS/VQA\'s flat CSV path\n        name: "",\n        confirmed: true,\n        answerText: "",\n        answerFrame: seed,          // confirmed-mode single answer (KIS/VQA)\n        answers: seed ? [seed] : [], // unconfirmed-mode ordered list, pre-seeded per spec ("at least 1 frame")\n        frameInfo: new Map(),        // "vid|n" -> {frame_idx, thumbnail_url} | "pending"\n        neighbourFrames: null,       // cached /api/export/neighbors result (grows with `neighboursShown`)\n        neighboursShown: PREVIEW_PAGE,\n        similarsShown: PREVIEW_PAGE,\n        dragIndex: null,\n        // TRAKE: no confirmed/unconfirmed distinction any more -- one\n        // per-video curation session (video + ordered event list, each a\n        // native frame_idx with its own add-time thumbnail) feeds a\n        // "Generate rows" call whose <=100 candidate sequences are cached\n        // here per video_id; a final merge step interleaves however many\n        // cached videos the human picked, in priority order, into one CSV.\n        // See freshTrakeState() below and the "TRAKE: curate one video\'s\n        // events..." section further down for the rest.\n        trake: freshTrakeState(),\n    };\n}\n\nfunction freshTrakeState() {\n    return {\n        videoId: null,\n        events: [],       // [{frame_idx, thumbnail}], in sequence order (event 1..N)\n        dragIndex: null,\n        videoEl: null,     // the curation panel\'s live <video>, for Add/capture\n        fps: 25,\n        // video_id -> {frameIdxs: [...], rows: [[f1..fN], ...]} -- one\n        // entry per "Generate rows" click; overwritten if regenerated for\n        // the same video_id.\n        cache: new Map(),\n        mergeOrder: [],    // video_ids, in merge priority order (checked ones only need be present)\n        mergeChecked: new Set(),\n        mergeDragIndex: null,\n    };\n}\n\nfunction frameKey(f) { return `${f.video_id}|${f.n}`; }\n\n// The Video ID/Frame ID typing boxes: `L21_V001`-shaped video id,\n// plain-integer frame number (the leading zeros in the "001" placeholder\n// are just display convention -- parseInt handles them fine either way).\n// The same parser serves both meanings the Frame ID box can have: a\n// keyframe n (KIS/VQA, resolved server-side via /api/export/frame) or a\n// raw native frame_idx (TRAKE, used directly, no resolution needed) --\n// both are just positive integers as typed. Return null on a malformed\n// box so the caller can show one "correct format" error rather than\n// letting a bad value reach the backend as a confusing 404/422.\nfunction parseVideoIdInput(str) {\n    const s = (str || "").trim();\n    return /^L\\d+_V\\d+$/i.test(s) ? s.toUpperCase() : null;\n}\nfunction parseFrameIdInput(str) {\n    const s = (str || "").trim();\n    if (!/^\\d+$/.test(s)) return null;\n    const n = parseInt(s, 10);\n    return n > 0 ? n : null;\n}\n\n// AIC submission naming: query-p2-<#>-<type>.csv, matching the real\n// submission/*.csv samples in the repo -- note VQA\'s type slug is "qa",\n// not "vqa".\nconst TYPE_SLUG = { KIS: "kis", VQA: "qa", TRAKE: "trake" };\nfunction queryFilename(queryType, name) {\n    return `query-p2-${name}-${TYPE_SLUG[queryType]}`;\n}\n\n// Kicks off a fetch for `f`\'s frame_idx if it isn\'t cached yet; callers\n// that already have a cached value use it directly when building their\n// HTML (see renderAnswerContent()) rather than going through here, so\n// `onReady` only ever fires asynchronously, after a genuine network round\n// trip -- never synchronously/re-entrantly. (It used to fire synchronously\n// for already-cached frames too, which re-entered renderAnswerContent()\n// from inside its own answers.forEach(), recursing once per already-cached\n// card; with more than a couple of cards that overflowed the call stack,\n// and since the throw happened inside this function\'s own promise chain,\n// its .catch() silently swallowed it and deleted the just-fetched cache\n// entry -- so a newly-added card\'s frame_idx never appeared, stuck on "…"\n// forever, with no visible error.)\nfunction ensureFrameInfo(s, f, onReady) {\n    const key = frameKey(f);\n    const cached = s.frameInfo.get(key);\n    if (cached) return; // already resolved-and-rendered, or a fetch is already in flight\n    s.frameInfo.set(key, "pending");\n    getExportFrame(f.video_id, f.n).then((info) => {\n        s.frameInfo.set(key, info);\n        onReady(info);\n    }).catch(() => { s.frameInfo.delete(key); });\n}\n\nexport function buildExportUI(container, trigger, { getCandidates, onDone }) {\n    const s = freshState(trigger);\n\n    const box = document.createElement("div");\n    box.className = "export-dialog";\n    box.innerHTML = `\n        <h2>Export CSV</h2>\n        <div class="status-banner" id="exp-status" style="display:none;"></div>\n        <div class="export-topbar" id="exp-topbar">\n          <div class="segmented" id="exp-segmented">\n            <button type="button" data-type="KIS">KIS</button>\n            <button type="button" data-type="VQA">VQA</button>\n            <button type="button" data-type="TRAKE">TRAKE</button>\n          </div>\n          <input type="number" id="exp-name" min="1" step="1" placeholder="Query no.">\n          <div class="export-change-fields" id="exp-change-fields">\n            <input type="text" id="exp-change-video" placeholder="Vid ID">\n            <input type="text" id="exp-change-frame" placeholder="Frame ID">\n            <button class="btn" id="exp-change-btn" type="button">Change</button>\n          </div>\n          <div class="export-actions">\n            <button class="btn" id="exp-cancel">Cancel</button>\n            <button class="btn btn-primary" id="exp-export">⬇ Export</button>\n          </div>\n        </div>\n        <div class="checkbox-row" id="exp-confirmed-row">\n          <input type="checkbox" id="exp-confirmed" checked>\n          <label for="exp-confirmed" style="margin:0;">Confirmed</label>\n        </div>\n        <div id="exp-flat-body">\n          <div class="export-answer-area">\n            <div id="exp-answer-content"></div>\n            <input type="text" id="exp-answer-text" placeholder="VQA answer" style="display:none;">\n          </div>\n        </div>\n        <div id="exp-trake-body" style="display:none;">\n          <div id="exp-trake-content"></div>\n        </div>\n        <div class="export-preview-area" id="exp-preview-area">\n          <div class="export-preview-section">\n            <div class="export-preview-header"><b>Neighbours</b> <span class="thumb-caption muted">(nearest keyframes by time)</span></div>\n            <div class="grid export-preview-grid" id="exp-nbr-grid"></div>\n            <button class="btn" id="exp-nbr-more">Show 12 more</button>\n          </div>\n          <div class="export-preview-section">\n            <div class="export-preview-header"><b>Similars</b> <span class="thumb-caption muted">(this query\'s ranked results)</span></div>\n            <div class="grid export-preview-grid" id="exp-sim-grid"></div>\n            <button class="btn" id="exp-sim-more">Show 12 more</button>\n          </div>\n        </div>`;\n    container.innerHTML = "";\n    container.append(box);\n\n    const el = (sel) => box.querySelector(sel);\n\n    // Status banner doubles as the validation-error banner (used throughout\n    // below) and the post-export confirmation (item: "leave it there with a\n    // message, user can reexport if they want" -- see #exp-export\'s handler,\n    // which shows success here rather than tearing the form down).\n    function showStatus(message, kind = "error") {\n        const banner = el("#exp-status");\n        banner.className = `status-banner ${kind}`;\n        banner.textContent = message;\n        banner.style.display = "block";\n    }\n    function clearStatus() {\n        el("#exp-status").style.display = "none";\n    }\n\n    function renderTypeVisibility() {\n        el("#exp-segmented").querySelectorAll("button").forEach((btn) => {\n            btn.classList.toggle("active", btn.dataset.type === s.queryType);\n        });\n        const isTrake = s.queryType === "TRAKE";\n        el("#exp-trake-body").style.display = isTrake ? "block" : "none";\n        el("#exp-flat-body").style.display = isTrake ? "none" : "block";\n        // No confirmed/unconfirmed distinction for TRAKE any more (see\n        // module docstring) -- the checkbox only means something for\n        // KIS/VQA.\n        el("#exp-confirmed-row").style.display = isTrake ? "none" : "flex";\n        // No Neighbours/Similars preview for TRAKE at all -- per spec,\n        // TRAKE events have no "similar" pool to search (only a picked\n        // frame\'s own keyframe/native-distance neighbours, which is what\n        // row generation already computes server-side); events come from\n        // the curation panel\'s video playback, the Frame ID box, or the\n        // seeding trigger only. renderPreview() itself skips fetching for\n        // TRAKE too, not just this visibility toggle.\n        el("#exp-preview-area").style.display = isTrake ? "none" : "flex";\n\n        // Frame ID/Change is repurposed for TRAKE, not hidden: a native\n        // frame number needs no keyframe lookup, so it means "add an\n        // event to the video currently being curated" instead of\n        // "replace/add the answer frame". Video ID is locked to that\n        // video (switching video is its own explicit action inside the\n        // TRAKE curation panel below) so the user can\'t typo an event\n        // into the wrong video\'s sequence.\n        el("#exp-change-frame").placeholder = isTrake ? "Frame ID (real frame)" : "Frame ID";\n        el("#exp-change-btn").textContent = isTrake ? "Add event" : "Change";\n        const changeVideo = el("#exp-change-video");\n        changeVideo.readOnly = isTrake;\n        changeVideo.value = isTrake ? (s.trake.videoId || "") : "";\n        changeVideo.title = isTrake ? "Switch curation video from the TRAKE panel below" : "";\n\n        // Same typing box either way -- unconfirmed mode used to disable\n        // this with an "LLM needed" placeholder (answering unconfirmed\n        // VQA queries was meant to be automated later), but that\'s no\n        // longer the plan: a human types the answer regardless of mode.\n        const isVqa = s.queryType === "VQA";\n        const answerText = el("#exp-answer-text");\n        answerText.style.display = isVqa ? "block" : "none";\n        answerText.disabled = false;\n        answerText.placeholder = "VQA answer";\n    }\n\n    function frameCardHtml(f, info, { removable = false, index = null } = {}) {\n        const thumb = info && info !== "pending"\n            ? `<img src="${info.thumbnail_url}" loading="lazy">`\n            : `<div class="thumb-missing">…</div>`;\n        const frameIdx = info && info !== "pending" ? info.frame_idx : "…";\n        const removeBtn = removable ? `<button class="icon-btn export-remove-btn" title="Remove" data-index="${index}">✕</button>` : "";\n        return `<div class="export-answer-card" ${removable ? `draggable="true" data-index="${index}"` : ""}>\n            <div class="thumb-wrap thumb-wrap-static">${thumb}</div>\n            <div class="thumb-caption"><b>${f.video_id}</b> · keyframe ${f.n}</div>\n            <div class="thumb-caption muted">real frame ${frameIdx}</div>\n            ${removeBtn}\n        </div>`;\n    }\n\n    function renderAnswerContent() {\n        const content = el("#exp-answer-content");\n        if (s.confirmed) {\n            if (!s.answerFrame) {\n                content.innerHTML = `<div class="status-banner info">No frame selected -- open this from a result card\'s ★ button.</div>`;\n                return;\n            }\n            const info = s.frameInfo.get(frameKey(s.answerFrame));\n            content.innerHTML = `<div class="export-answer-list">${frameCardHtml(s.answerFrame, info)}</div>`;\n            ensureFrameInfo(s, s.answerFrame, () => renderAnswerContent());\n        } else {\n            if (!s.answers.length) {\n                content.innerHTML = `<div class="status-banner info">Add at least one frame from the preview below.</div>`;\n            } else {\n                content.innerHTML = `<div class="export-answer-list">${s.answers.map((f, i) => {\n                    const info = s.frameInfo.get(frameKey(f));\n                    return frameCardHtml(f, info, { removable: true, index: i });\n                }).join("")}</div>`;\n                s.answers.forEach((f) => ensureFrameInfo(s, f, () => renderAnswerContent()));\n            }\n            wireAnswerDnd();\n        }\n    }\n\n    function wireAnswerDnd() {\n        for (const card of el("#exp-answer-content").querySelectorAll(".export-answer-card")) {\n            const i = Number(card.dataset.index);\n            card.addEventListener("dragstart", () => { s.dragIndex = i; });\n            card.addEventListener("dragover", (e) => e.preventDefault());\n            card.addEventListener("drop", (e) => {\n                e.preventDefault();\n                if (s.dragIndex === null || s.dragIndex === i) return;\n                const [moved] = s.answers.splice(s.dragIndex, 1);\n                s.answers.splice(i, 0, moved);\n                s.dragIndex = null;\n                renderAnswerContent();\n            });\n            const removeBtn = card.querySelector(".export-remove-btn");\n            if (removeBtn) removeBtn.onclick = () => {\n                s.answers.splice(i, 1);\n                renderAnswerContent();\n            };\n        }\n    }\n\n    function isInAnswers(f) {\n        return s.answers.some((a) => a.video_id === f.video_id && a.n === f.n);\n    }\n\n    function addToAnswers(f) {\n        if (isInAnswers(f)) return;\n        s.answers.push({ video_id: f.video_id, n: f.n });\n        // Mirror into the confirmed single-frame slot only while there\'s\n        // still exactly one frame in the unconfirmed list -- once a second\n        // is added, "the" answer frame is ambiguous, so stop syncing\n        // rather than guess which one confirmed mode should show.\n        if (s.answers.length === 1) s.answerFrame = s.answers[0];\n        renderAnswerContent();\n        renderPreview();\n    }\n\n    function previewCardHtml(f, { addable, replaceable }) {\n        const already = addable && isInAnswers(f);\n        const isCurrent = replaceable && s.answerFrame && s.answerFrame.video_id === f.video_id && s.answerFrame.n === f.n;\n        const addBtn = addable\n            ? `<button class="icon-btn export-add-btn${already ? " added" : ""}" title="${already ? "Already added" : "Add to answer(s)"}" data-video-id="${f.video_id}" data-n="${f.n}">${already ? "✓" : "+"}</button>`\n            : "";\n        // Confirmed mode: picking a preview frame replaces the single\n        // answer frame instead of adding to a list. Reuses .export-add-btn\'s\n        // CSS (same corner position, same .added accent) and is told apart\n        // from the add button by data-replace, not class.\n        const replaceBtn = replaceable\n            ? `<button class="icon-btn export-add-btn${isCurrent ? " added" : ""}" title="${isCurrent ? "Current answer frame" : "Use as answer frame"}" data-video-id="${f.video_id}" data-n="${f.n}" data-replace="1"${isCurrent ? " disabled" : ""}>${isCurrent ? "✓" : "⇄"}</button>`\n            : "";\n        return `<div class="thumb-cell">\n            <div class="thumb-wrap thumb-wrap-static"><img src="${f.thumbnail_url}" loading="lazy"></div>\n            <div class="thumb-caption"><b>${f.video_id}</b> · frame ${f.n}</div>\n            ${addBtn}${replaceBtn}\n        </div>`;\n    }\n\n    // KIS/VQA only -- TRAKE has no Neighbours/Similars preview at all (see\n    // renderTypeVisibility\'s #exp-preview-area toggle and module\n    // docstring: a TRAKE pick\'s only "similar" pool is what row generation\n    // already computes server-side, not something to browse here).\n    async function renderPreview() {\n        if (s.queryType === "TRAKE") return;\n\n        const addable = !s.confirmed;\n        const replaceable = s.confirmed;\n\n        // Neighbours -- nearest keyframes by time to the trigger frame.\n        const nbrGrid = el("#exp-nbr-grid");\n        if (s.trigger.kind !== "flat") {\n            nbrGrid.innerHTML = `<div class="status-banner info">No source frame to find neighbours of.</div>`;\n        } else {\n            if (!s.neighbourFrames || s.neighbourFrames.length < s.neighboursShown) {\n                nbrGrid.innerHTML = `<div class="status-banner info">Loading…</div>`;\n                try {\n                    const data = await getExportNeighbors(s.trigger.video_id, s.trigger.n, s.neighboursShown);\n                    s.neighbourFrames = data.frames;\n                } catch (e) {\n                    nbrGrid.innerHTML = `<div class="status-banner error">${e.message}</div>`;\n                    return;\n                }\n            }\n            const frames = s.neighbourFrames.slice(0, s.neighboursShown).map((f) => ({ ...f, video_id: s.trigger.video_id }));\n            nbrGrid.innerHTML = frames.map((f) => previewCardHtml(f, { addable, replaceable })).join("") || `<div class="status-banner info">No neighbours found.</div>`;\n            el("#exp-nbr-more").style.display = s.neighbourFrames.length >= s.neighboursShown ? "block" : "none";\n        }\n\n        // Similars -- the query\'s own already-fetched, already-ranked results.\n        // These may be TRAKE-shaped ({video_id, events}, no .n) if the\n        // opener tab\'s last search was a real TRAKE search -- previewCardHtml\n        // needs a flat {video_id, n} shape, so fall back to a plain message\n        // rather than rendering broken cards.\n        const candidates = getCandidates();\n        const simGrid = el("#exp-sim-grid");\n        if (candidates.length && !("n" in candidates[0])) {\n            simGrid.innerHTML = `<div class="status-banner info">Last search wasn\'t a flat-result signal -- no Similars to preview.</div>`;\n            el("#exp-sim-more").style.display = "none";\n        } else {\n            const similars = candidates.slice(0, s.similarsShown);\n            simGrid.innerHTML = similars.map((c) => previewCardHtml(c, { addable, replaceable }))\n                .join("") || `<div class="status-banner info">No results from the last search.</div>`;\n            el("#exp-sim-more").style.display = candidates.length > s.similarsShown ? "block" : "none";\n        }\n\n        if (addable) {\n            box.querySelectorAll(".export-add-btn:not([data-replace])").forEach((btn) => {\n                btn.onclick = () => addToAnswers({ video_id: btn.dataset.videoId, n: Number(btn.dataset.n) });\n            });\n        }\n        if (replaceable) {\n            box.querySelectorAll(".export-add-btn[data-replace]").forEach((btn) => {\n                btn.onclick = () => applyChangedFrame({ video_id: btn.dataset.videoId, n: Number(btn.dataset.n) });\n            });\n        }\n    }\n\n    // Item 4 (pick a preview frame to replace) and item 5 (type a video/\n    // frame id and hit Change) both funnel through here: confirmed mode\n    // replaces the single answer frame, unconfirmed mode adds to the answer\n    // list (same as the preview\'s own "+" button).\n    //\n    // Confirmed\'s answerFrame and unconfirmed\'s answers list also get\n    // synced here -- a confirmed-mode edit collapses the answers list down\n    // to that one frame, since at that point there\'s exactly one frame in\n    // play and both views should agree on it (otherwise toggling Confirmed\n    // off would silently revert to whatever the tab was originally seeded\n    // with instead of the just-changed frame).\n    function applyChangedFrame(f) {\n        if (s.confirmed) {\n            if (s.answerFrame && frameKey(s.answerFrame) === frameKey(f)) return;\n            s.answerFrame = f;\n            s.answers = [f];\n            renderAnswerContent();\n            renderPreview();\n        } else {\n            addToAnswers(f);\n        }\n    }\n\n    // --- TRAKE: curate one video\'s events -> cache its generated rows ->\n    // merge however many cached videos into the final export -----------\n\n    function fmtTime(t) {\n        const mm = String(Math.floor(t / 60)).padStart(2, "0");\n        const ss = (t % 60).toFixed(2).padStart(5, "0");\n        return `${mm}:${ss}`;\n    }\n\n    // Grabs a JPEG data URL of whatever frame `video` is showing right\n    // now -- this is the "cache the thumbnail at add-time" half of the\n    // spec, since a raw native frame has no existing thumbnail file to\n    // point at the way a keyframe does. The video element is same-origin\n    // (served from this app\'s own /media/video mount), so the canvas\n    // isn\'t tainted; still guarded in case a frame isn\'t decoded yet.\n    function captureVideoThumbnail(video) {\n        try {\n            const w = video.videoWidth || 320, h = video.videoHeight || 180;\n            const canvas = document.createElement("canvas");\n            canvas.width = 160;\n            canvas.height = Math.round(160 * (h / w));\n            canvas.getContext("2d").drawImage(video, 0, 0, canvas.width, canvas.height);\n            return canvas.toDataURL("image/jpeg", 0.7);\n        } catch (e) {\n            return null;\n        }\n    }\n\n    // Loads (or switches the curation panel to) a video: fetches playback\n    // info and builds a fresh <video>. Switching to a *different* video\n    // than the one currently being curated starts a clean event list --\n    // any cache entry already generated for either video is untouched\n    // (cache entries persist independently of what\'s in the live curation\n    // panel, see generateRowsForCurationVideo()).\n    async function loadCurationVideo(videoId) {\n        videoId = (videoId || "").trim().toUpperCase();\n        if (!videoId) return;\n        if (videoId !== s.trake.videoId) {\n            s.trake.videoId = videoId;\n            s.trake.events = [];\n            renderEventList();\n        }\n        clearStatus();\n        renderTypeVisibility(); // keeps the topbar\'s locked Video ID display in sync\n        const wrap = el("#trake-video-wrap");\n        if (wrap) wrap.innerHTML = `<div class="status-banner info">Loading…</div>`;\n        try {\n            const data = await getPlayback(videoId);\n            if (el("#trake-video-wrap") !== wrap) return; // panel torn down mid-fetch (query type switched away)\n            s.trake.fps = data.fps;\n            wrap.innerHTML = "";\n            if (data.has_video) {\n                const video = document.createElement("video");\n                video.src = data.video_url;\n                video.controls = true;\n                video.preload = "metadata";\n                video.playsInline = true;\n                wrap.append(video);\n                s.trake.videoEl = video;\n                const timer = el("#trake-cur-timer");\n                video.addEventListener("timeupdate", () => {\n                    timer.textContent = `${fmtTime(video.currentTime)} · frame ${Math.round(video.currentTime * s.trake.fps)}`;\n                });\n                video.onerror = () => {\n                    s.trake.videoEl = null;\n                    if (wrap) wrap.innerHTML = `<div class="status-banner warn">Video file stream unavailable for ${videoId}. Use keyframe export instead.</div>`;\n                };\n            } else {\n                s.trake.videoEl = null;\n                if (wrap) wrap.innerHTML = `<div class="status-banner warn">Video file .mp4 not found on disk for ${videoId}. Use keyframe export instead.</div>`;\n            }\n        } catch (e) {\n            s.trake.videoEl = null;\n            if (wrap) wrap.innerHTML = `<div class="status-banner error">${e.message}</div>`;\n        }\n    }\n\n    // Adds one event to the video currently being curated -- from the\n    // inline "Add current frame" button (f.thumbnail already captured) or\n    // the repurposed Frame ID/Change row (raw frame_idx, no thumbnail).\n    // Enforces the one hard constraint: a TRAKE export row is exactly one\n    // video, so a frame from a different video is rejected rather than\n    // silently starting a second, unrepresentable sequence.\n    function addTrakeEvent(f) {\n        if (s.trake.videoId && f.video_id !== s.trake.videoId) {\n            showStatus(`Currently curating ${s.trake.videoId} -- this frame is from a different video. Switch videos above first if you meant to add it there.`);\n            return false;\n        }\n        if (!s.trake.videoId) s.trake.videoId = f.video_id;\n        s.trake.events.push({ frame_idx: f.frame_idx, thumbnail: f.thumbnail ?? null });\n        clearStatus();\n        renderEventList();\n        return true;\n    }\n\n    // Resolves a keyframe n (not a raw frame_idx) to its real frame_idx +\n    // thumbnail before adding -- used to seed the curation panel from a\n    // "flat" trigger (any non-TRAKE signal\'s ★, which only carries n, no\n    // frame_idx) via the same lookup the rest of the app already does.\n    function addTrakeEventFromN(videoId, n) {\n        if (s.trake.videoId && videoId !== s.trake.videoId) {\n            showStatus(`Currently curating ${s.trake.videoId} -- this frame is from a different video. Switch videos above first if you meant to add it there.`);\n            return;\n        }\n        getExportFrame(videoId, n).then((info) => {\n            addTrakeEvent({ video_id: videoId, frame_idx: info.frame_idx, thumbnail: info.thumbnail_url });\n        }).catch((e) => showStatus(e.message));\n    }\n\n    function removeTrakeEvent(i) {\n        s.trake.events.splice(i, 1);\n        renderEventList();\n    }\n    function moveTrakeEvent(from, to) {\n        const [ev] = s.trake.events.splice(from, 1);\n        s.trake.events.splice(to, 0, ev);\n        renderEventList();\n    }\n    function wireTrakeEventDnd(list) {\n        for (const row of list.querySelectorAll(".trake-event-row")) {\n            const i = Number(row.dataset.index);\n            row.addEventListener("dragstart", () => { s.trake.dragIndex = i; });\n            row.addEventListener("dragover", (e) => e.preventDefault());\n            row.addEventListener("drop", (e) => {\n                e.preventDefault();\n                if (s.trake.dragIndex === null || s.trake.dragIndex === i) return;\n                moveTrakeEvent(s.trake.dragIndex, i);\n                s.trake.dragIndex = null;\n            });\n            const removeBtn = row.querySelector(".export-remove-btn");\n            if (removeBtn) removeBtn.onclick = () => removeTrakeEvent(i);\n        }\n    }\n\n    // Only the event list -- never the video element or the cache panel --\n    // so adding/removing/reordering an event never interrupts playback.\n    function renderEventList() {\n        const list = el("#trake-event-list");\n        if (!list) return;\n        if (!s.trake.events.length) {\n            list.innerHTML = `<div class="status-banner info">No events yet -- play the video and click "Add current frame", or add one from a preview card\'s "+" / the Frame ID box above.</div>`;\n            return;\n        }\n        list.innerHTML = s.trake.events.map((e, i) => `\n            <div class="trake-event-row" draggable="true" data-index="${i}">\n                ${e.thumbnail\n                    ? `<div class="thumb-wrap thumb-wrap-static"><img src="${e.thumbnail}" loading="lazy"></div>`\n                    : `<div class="thumb-missing">no preview</div>`}\n                <div class="trake-event-fields">\n                    <div class="thumb-caption"><b>E${i + 1}</b></div>\n                    <div class="thumb-caption muted">frame ${e.frame_idx}</div>\n                </div>\n                <button class="icon-btn export-remove-btn" title="Remove" data-index="${i}">✕</button>\n            </div>`).join("");\n        wireTrakeEventDnd(list);\n    }\n\n    // POSTs this video\'s curated {video_id, frame_idxs} to\n    // /api/export/trake-rows and stashes the <=100 returned candidate\n    // sequences client-side, keyed by video_id -- repeatable for as many\n    // candidate videos as the human wants to compare (each just adds/\n    // overwrites its own cache entry, see module docstring). Newly cached\n    // (or re-cached) videos default to checked-and-appended into the\n    // merge priority order, last -- keeps a freshly regenerated video in\n    // whatever priority slot it already had instead of bumping it to the\n    // front.\n    async function generateRowsForCurationVideo() {\n        if (!s.trake.videoId || !s.trake.events.length) {\n            showStatus("Add at least one event before generating rows.");\n            return;\n        }\n        const btn = el("#trake-generate-btn");\n        if (btn) btn.disabled = true;\n        try {\n            const frameIdxs = s.trake.events.map((e) => e.frame_idx);\n            const thumbnails = s.trake.events.map((e) => e.thumbnail);\n            const data = await getTrakeRows(s.trake.videoId, frameIdxs, 100);\n            s.trake.cache.set(s.trake.videoId, { frameIdxs, thumbnails, rows: data.rows });\n            if (!s.trake.mergeOrder.includes(s.trake.videoId)) s.trake.mergeOrder.push(s.trake.videoId);\n            s.trake.mergeChecked.add(s.trake.videoId);\n            showStatus(`✓ Cached ${data.rows.length} rows for ${s.trake.videoId}. Curate another video, or check it below and Export.`, "info");\n            renderCacheList();\n        } catch (e) {\n            showStatus(e.message);\n        } finally {\n            if (btn) btn.disabled = false;\n        }\n    }\n\n    function removeFromCache(videoId) {\n        s.trake.cache.delete(videoId);\n        s.trake.mergeOrder = s.trake.mergeOrder.filter((v) => v !== videoId);\n        s.trake.mergeChecked.delete(videoId);\n        renderCacheList();\n    }\n\n    function wireCacheDnd(list) {\n        for (const row of list.querySelectorAll(".trake-cache-row")) {\n            const vid = row.dataset.videoId;\n            row.addEventListener("dragstart", () => { s.trake.mergeDragIndex = s.trake.mergeOrder.indexOf(vid); });\n            row.addEventListener("dragover", (e) => e.preventDefault());\n            row.addEventListener("drop", (e) => {\n                e.preventDefault();\n                const from = s.trake.mergeDragIndex;\n                const to = s.trake.mergeOrder.indexOf(vid);\n                if (from === null || from === to) return;\n                const [moved] = s.trake.mergeOrder.splice(from, 1);\n                s.trake.mergeOrder.splice(to, 0, moved);\n                s.trake.mergeDragIndex = null;\n                renderCacheList();\n            });\n        }\n    }\n\n    function renderCacheList() {\n        const list = el("#trake-cache-list");\n        if (!list) return;\n        if (!s.trake.cache.size) {\n            list.innerHTML = `<div class="status-banner info">Nothing cached yet -- curate a video above, then click "Generate rows".</div>`;\n            return;\n        }\n        list.innerHTML = s.trake.mergeOrder.map((vid) => {\n            const entry = s.trake.cache.get(vid);\n            if (!entry) return "";\n            const checked = s.trake.mergeChecked.has(vid);\n            return `<div class="trake-cache-row" draggable="true" data-video-id="${vid}">\n                <input type="checkbox" class="trake-cache-check" data-video-id="${vid}" ${checked ? "checked" : ""}>\n                <span class="thumb-caption"><b>${vid}</b> <span class="muted">· ${entry.rows.length} rows · ${entry.frameIdxs.length} events</span></span>\n                <button class="icon-btn export-remove-btn" title="Remove from cache" data-video-id="${vid}">✕</button>\n            </div>`;\n        }).join("");\n        list.querySelectorAll(".trake-cache-check").forEach((cb) => {\n            cb.onchange = () => {\n                const vid = cb.dataset.videoId;\n                if (cb.checked) s.trake.mergeChecked.add(vid); else s.trake.mergeChecked.delete(vid);\n            };\n        });\n        list.querySelectorAll(".trake-cache-row .export-remove-btn").forEach((btn) => {\n            btn.onclick = () => removeFromCache(btn.dataset.videoId);\n        });\n        wireCacheDnd(list);\n    }\n\n    // Client-side only -- "no CSV parsing, no re-reading files" per spec.\n    // Each checked video\'s own row 1 (its curated pick) goes first, in\n    // priority order, then row 2/row 3/... round-robin in that same\n    // order until the cap is hit or every cached video\'s rows are spent.\n    // This mirrors the rest of the app\'s export tiers (one clean row per\n    // hypothesis first, hedges/fillers after) while keeping the highest-\n    // priority video\'s own pick at rank 1, which is what R@1 rewards.\n    function mergeTrakeCache(maxRows) {\n        const selected = s.trake.mergeOrder.filter((vid) => s.trake.mergeChecked.has(vid) && s.trake.cache.has(vid));\n        const rows = [];\n        for (let k = 0; rows.length < maxRows; k++) {\n            let any = false;\n            for (const vid of selected) {\n                const entry = s.trake.cache.get(vid);\n                if (k < entry.rows.length) {\n                    rows.push({ video_id: vid, frame_idxs: entry.rows[k] });\n                    any = true;\n                    if (rows.length >= maxRows) break;\n                }\n            }\n            if (!any) break;\n        }\n        return rows;\n    }\n\n    let trakeSkeletonBuilt = false;\n\n    // Builds the panel\'s static DOM once (the video element and cache\n    // list are updated in place afterward, by renderEventList()/\n    // renderCacheList(), never rebuilt wholesale -- rebuilding on every\n    // state change would tear down and restart the <video> mid-playback).\n    function ensureTrakeSkeleton() {\n        if (trakeSkeletonBuilt) return;\n        trakeSkeletonBuilt = true;\n        el("#exp-trake-content").innerHTML = `\n            <div class="trake-curate-panel">\n              <div class="trake-toprow">\n                <input type="text" id="trake-load-video" placeholder="Video ID e.g. L21_V001">\n                <button class="btn" id="trake-load-btn" type="button">Load / switch</button>\n                <span id="trake-cur-timer" class="playback-timer">--:-- · frame --</span>\n                <button class="btn btn-primary" id="trake-add-btn" type="button">+ Add current frame as event</button>\n              </div>\n              <div class="trake-main-row">\n                <div class="trake-video-col" id="trake-video-wrap">\n                  <div class="status-banner info">Load a video above, or open this tab from a result card\'s ★.</div>\n                </div>\n                <div class="trake-events-col">\n                  <div class="thumb-caption muted" style="margin-bottom:0.4rem;">Events, in sequence order -- drag to reorder, ✕ to remove:</div>\n                  <div class="trake-event-list" id="trake-event-list"></div>\n                  <button class="btn btn-primary" id="trake-generate-btn" type="button" style="margin-top:0.6rem;">Generate rows for this video</button>\n                </div>\n              </div>\n            </div>\n            <hr class="divider">\n            <div class="trake-cache-panel">\n              <div class="thumb-caption" style="margin-bottom:0.4rem;"><b>Cached videos</b> <span class="muted">(check to include in the merged export, drag to set priority order)</span></div>\n              <div id="trake-cache-list"></div>\n            </div>`;\n\n        el("#trake-load-btn").onclick = () => loadCurationVideo(el("#trake-load-video").value);\n        el("#trake-add-btn").onclick = () => {\n            if (!s.trake.videoEl) { showStatus("No video loaded to capture a frame from."); return; }\n            const video = s.trake.videoEl;\n            const frame_idx = Math.round(video.currentTime * s.trake.fps);\n            addTrakeEvent({ video_id: s.trake.videoId, frame_idx, thumbnail: captureVideoThumbnail(video) });\n        };\n        el("#trake-generate-btn").onclick = generateRowsForCurationVideo;\n    }\n\n    function renderTrakeContent() {\n        ensureTrakeSkeleton();\n        renderEventList();\n        renderCacheList();\n    }\n\n    // --- wiring ---------------------------------------------------------\n\n    el("#exp-segmented").querySelectorAll("button").forEach((btn) => {\n        btn.onclick = () => {\n            s.queryType = btn.dataset.type;\n            renderTypeVisibility();\n            renderTrakeContent();\n            renderAnswerContent();\n            renderPreview(); // replaceable depends on queryType (TRAKE has no single answer frame)\n        };\n    });\n    el("#exp-name").oninput = (e) => { s.name = e.target.value; };\n    el("#exp-answer-text").oninput = (e) => { s.answerText = e.target.value; };\n    el("#exp-confirmed").onchange = (e) => {\n        s.confirmed = e.target.checked;\n        renderTypeVisibility();\n        renderAnswerContent();\n        renderPreview();\n        renderTrakeContent();\n    };\n    el("#exp-nbr-more").onclick = () => { s.neighboursShown += PREVIEW_PAGE; renderPreview(); };\n    el("#exp-sim-more").onclick = () => { s.similarsShown += PREVIEW_PAGE; renderPreview(); };\n    el("#exp-cancel").onclick = () => onDone("cancel");\n\n    // Typed "Video ID" / "Frame ID" boxes + Change/Add event button.\n    // KIS/VQA: same destination as the preview-pick (applyChangedFrame),\n    // but reaches an arbitrary frame not necessarily in either preview\n    // list -- verifies the frame actually exists (via /api/export/frame,\n    // n-based) before applying, so a typo lands as one clear error rather\n    // than a broken export.\n    // TRAKE: "Frame ID" is a raw native frame number, not a keyframe n,\n    // added to whatever video the curation panel below is already on --\n    // the (read-only) Video ID box is just a reminder of that, not a\n    // second way to pick the video (see the panel\'s own Load/switch row\n    // for that). No backend round-trip needed at all.\n    el("#exp-change-btn").onclick = async () => {\n        clearStatus();\n\n        if (s.queryType === "TRAKE") {\n            if (!s.trake.videoId) {\n                showStatus("Load a video in the TRAKE panel below first.");\n                return;\n            }\n            const num = parseFrameIdInput(el("#exp-change-frame").value);\n            if (!num) {\n                showStatus("Enter a real frame number.");\n                return;\n            }\n            if (addTrakeEvent({ video_id: s.trake.videoId, frame_idx: num })) {\n                el("#exp-change-frame").value = "";\n            }\n            return;\n        }\n\n        const videoId = parseVideoIdInput(el("#exp-change-video").value);\n        const num = parseFrameIdInput(el("#exp-change-frame").value);\n        if (!videoId || !num) {\n            showStatus("Enter a valid video id (e.g. L21_V001) and frame id (e.g. 001).");\n            return;\n        }\n\n        const btn = el("#exp-change-btn");\n        btn.disabled = true;\n        try {\n            await getExportFrame(videoId, num); // throws if that frame doesn\'t exist for that video\n            applyChangedFrame({ video_id: videoId, n: num });\n            el("#exp-change-video").value = "";\n            el("#exp-change-frame").value = "";\n        } catch (e) {\n            showStatus(e.message);\n        } finally {\n            btn.disabled = false;\n        }\n    };\n\n    el("#exp-export").onclick = async () => {\n        clearStatus();\n\n        if (!s.name) {\n            showStatus("Enter a query number.");\n            return;\n        }\n\n        if (s.queryType === "TRAKE") {\n            // Client-side merge of the per-video cache -- no candidates/\n            // confirmed/answers body to build, unlike KIS/VQA below.\n            const merged = mergeTrakeCache(100);\n            if (!merged.length) {\n                showStatus("Nothing to export -- curate a video, click \\"Generate rows\\", then check it below.");\n                return;\n            }\n            const filename = queryFilename("TRAKE", s.name);\n            const exportBtn = el("#exp-export");\n            exportBtn.disabled = true;\n            try {\n                await writeTrakeCsv(merged, filename);\n                showStatus(`✓ Exported ${filename}.csv (${merged.length} rows) -- you can export again from here if needed.`, "info");\n            } catch (e) {\n                showStatus(e.message);\n            } finally {\n                exportBtn.disabled = false;\n            }\n            return;\n        }\n\n        if (s.confirmed && !s.answerFrame) {\n            showStatus("No confirmed frame -- open this from a result card.");\n            return;\n        }\n        if (!s.confirmed && !s.answers.length) {\n            showStatus("Add at least one answer frame from the preview.");\n            return;\n        }\n        const body = {\n            query_type: s.queryType,\n            mode: s.confirmed ? "confirmed" : "unconfirmed",\n            candidates: getCandidates(),\n            confirmed: s.confirmed ? s.answerFrame : null,\n            answers: s.confirmed ? [] : s.answers,\n            answer: s.answerText,\n            neighbour_count: NEIGHBOUR_COUNT_EXPORT,\n            filename: queryFilename(s.queryType, s.name),\n        };\n\n        const exportBtn = el("#exp-export");\n        exportBtn.disabled = true;\n        try {\n            await exportCsv(body);\n            // Left in place, not closed/torn down -- the form stays exactly\n            // as it was, so the user can immediately re-export (a new query\n            // #, a tweaked frame, ...) without reopening this tab.\n            showStatus(`✓ Exported ${body.filename}.csv -- you can export again from here if needed.`, "info");\n        } catch (e) {\n            showStatus(e.message);\n        } finally {\n            exportBtn.disabled = false;\n        }\n    };\n\n    // A "frame" trigger (from video playback) has no keyframe n at all --\n    // KIS/VQA\'s flat CSV path needs one (backend\'s frame_idx_for_n), so\n    // those two types aren\'t usable here. TRAKE stays the only option.\n    if (trigger.kind === "frame") {\n        for (const type of ["KIS", "VQA"]) {\n            const btn = el(`#exp-segmented button[data-type="${type}"]`);\n            btn.disabled = true;\n            btn.title = "Needs a keyframe-backed frame -- use TRAKE for a native frame from playback.";\n        }\n    }\n\n    renderTypeVisibility();\n    renderAnswerContent();\n    renderPreview();\n    renderTrakeContent();\n\n    // Seed the curation panel straight from `trigger` when it carries a\n    // video/frame of its own -- any signal\'s result card, a real TRAKE\n    // candidate\'s own matched events, or a raw playback frame can all\n    // start (or extend) a TRAKE sequence, not just a real TRAKE search\n    // (see module docstring). Not limited to when queryType actually\n    // starts on TRAKE -- switching to TRAKE later still finds the panel\n    // already seeded.\n    // loadCurationVideo is called right after kicking the event-seeding\n    // off (not awaited first) so the video starts loading immediately\n    // rather than waiting on the frame-info round trip(s) below; its own\n    // videoId!==s.trake.videoId check still resets s.trake.events first,\n    // but that\'s a no-op here since freshState() always starts empty.\n    let seedVideoId = null;\n    if (trigger.kind === "trake") {\n        // Resolved in parallel but applied in original event order (matters\n        // here, unlike a lone addTrakeEventFromN call elsewhere) -- several\n        // concurrent fetches racing straight into addTrakeEvent could\n        // otherwise land E2 before E1 depending on which response arrives\n        // first.\n        seedVideoId = trigger.candidate.video_id;\n        const matched = trigger.candidate.events.filter((e) => e.matched);\n        Promise.all(matched.map((e) => getExportFrame(seedVideoId, e.n).catch(() => null))).then((infos) => {\n            for (const info of infos) {\n                if (info) addTrakeEvent({ video_id: seedVideoId, frame_idx: info.frame_idx, thumbnail: info.thumbnail_url });\n            }\n        });\n    } else if (trigger.kind === "flat") {\n        seedVideoId = trigger.video_id;\n        addTrakeEventFromN(trigger.video_id, trigger.n);\n    } else if (trigger.kind === "frame") {\n        seedVideoId = trigger.video_id;\n        addTrakeEvent({ video_id: trigger.video_id, frame_idx: trigger.frame_idx });\n    }\n    if (seedVideoId) loadCurationVideo(seedVideoId);\n}\n', encoding="utf-8")
(WORKSPACE_DIR / "frontend/css/style.css").write_text('/* frontend/css/style.css -- full layout (Streamlit\'s own chrome is gone, so\n   this covers page structure too) plus the two thumbnail-hover rules\n   ported verbatim from ui/app.py:1186-1220. */\n\n:root {\n    --bg: #ffffff;\n    --bg-sidebar: #f0f2f6;\n    --text: #262730;\n    --text-muted: #6b7280;\n    --border: #d6d9de;\n    --accent: #ff4b4b;\n    --accent-soft: #ffe8e8;\n    --radius: 6px;\n}\n\n* { box-sizing: border-box; }\n\nhtml, body {\n    margin: 0;\n    padding: 0;\n    font-family: "Source Sans Pro", -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;\n    color: var(--text);\n    background: var(--bg);\n}\n\n.layout {\n    display: flex;\n    min-height: 100vh;\n}\n\n/* ---------------------------------------------------------------------- */\n/* Sidebar */\n/* ---------------------------------------------------------------------- */\n\n.sidebar {\n    width: 320px;\n    flex: none;\n    background: var(--bg-sidebar);\n    border-right: 1px solid var(--border);\n    padding: 1.25rem 1rem;\n    overflow-y: auto;\n    height: 100vh;\n    position: sticky;\n    top: 0;\n}\n\n.sidebar h2 {\n    margin: 0 0 0.75rem;\n    font-size: 1.1rem;\n}\n\n.signal-row {\n    display: flex;\n    flex-wrap: wrap;\n    gap: 0.4rem;\n    margin-bottom: 0.6rem;\n}\n\n.signal-btn {\n    display: flex;\n    align-items: center;\n    justify-content: center;\n    width: 42px;\n    height: 36px;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    background: #fff;\n    cursor: pointer;\n    font-size: 1.1rem;\n}\n\n.signal-btn:hover { background: var(--accent-soft); }\n.signal-btn.active { border-color: var(--accent); background: var(--accent-soft); }\n.signal-btn:disabled { opacity: 0.35; cursor: not-allowed; }\n\n.sidebar label {\n    display: block;\n    font-size: 0.85rem;\n    margin: 0.75rem 0 0.25rem;\n    color: var(--text-muted);\n}\n\n.sidebar textarea,\n.sidebar input[type="text"],\n.sidebar input[type="number"] {\n    width: 100%;\n    font-family: inherit;\n    font-size: 0.9rem;\n    padding: 0.4rem 0.5rem;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    background: #fff;\n    color: var(--text);\n    resize: vertical;\n}\n\n.sidebar textarea { min-height: 64px; }\n\n.scope-row { display: flex; gap: 0.5rem; }\n.scope-row > div { flex: 1; }\n\n.checkbox-row {\n    display: flex;\n    align-items: center;\n    gap: 0.4rem;\n    font-size: 0.88rem;\n    margin: 0.35rem 0;\n}\n.checkbox-row input { margin: 0; }\n\n.image-query-preview {\n    display: flex;\n    align-items: center;\n    gap: 0.5rem;\n    margin: 0.5rem 0;\n}\n.image-query-preview img {\n    width: 60px;\n    height: 60px;\n    object-fit: cover;\n    border-radius: var(--radius);\n    border: 1px solid var(--border);\n}\n\nbutton {\n    font-family: inherit;\n    cursor: pointer;\n}\n\n.btn {\n    padding: 0.4rem 0.75rem;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    background: #fff;\n    font-size: 0.85rem;\n}\n.btn:hover { background: var(--accent-soft); }\n.btn-primary { background: var(--accent); border-color: var(--accent); color: #fff; }\n.btn-primary:hover { filter: brightness(1.05); }\n\n/* ---------------------------------------------------------------------- */\n/* Main content */\n/* ---------------------------------------------------------------------- */\n\n.main {\n    flex: 1;\n    padding: 1.5rem 2rem;\n    min-width: 0;\n}\n\n.main h1 { margin-top: 0; }\n.main h2 { margin: 1.5rem 0 0.75rem; }\n\n.status-banner {\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    padding: 0.6rem 1rem;\n    margin-bottom: 1rem;\n    font-size: 0.9rem;\n}\n.status-banner.info { background: #e8f0fe; border-color: #b3d1ff; }\n.status-banner.warn { background: #fff8e1; border-color: #ffe08a; }\n.status-banner.error { background: #fdeaea; border-color: #f3b8b8; }\n\n.grid {\n    display: grid;\n    grid-template-columns: repeat(5, 1fr);\n    gap: 0.75rem 1rem;\n    margin-bottom: 1rem;\n}\n\n@media (max-width: 1100px) {\n    .grid { grid-template-columns: repeat(3, 1fr); }\n}\n\n.thumb-cell { font-size: 0.82rem; }\n\n.thumb-wrap { position: relative; overflow: visible; z-index: 1; line-height: 0; }\n.thumb-wrap img {\n    width: 100%; display: block; border-radius: 4px;\n    transition: transform 0.15s ease-out, box-shadow 0.15s ease-out;\n    transform-origin: center center;\n}\n.thumb-wrap:hover { z-index: 100; }\n.thumb-wrap:hover img {\n    transform: scale(2.2);\n    box-shadow: 0 12px 32px rgba(0,0,0,0.45);\n    position: relative;\n}\n/* TRAKE result cards with only 1-2 events: the zoom-on-hover above is more\n   distracting than useful with so few thumbnails already at a decent size,\n   so it\'s suppressed for those (kept for 3+ event cards). */\n.thumb-wrap-static:hover { z-index: 1; }\n.thumb-wrap-static:hover img { transform: none; box-shadow: none; position: static; }\n\n.thumb-missing {\n    aspect-ratio: 1;\n    display: flex;\n    align-items: center;\n    justify-content: center;\n    background: #f3f4f6;\n    border-radius: 4px;\n    color: var(--text-muted);\n    font-size: 0.75rem;\n}\n\n.thumb-caption { color: var(--text); margin-top: 0.25rem; }\n.thumb-caption.muted { color: var(--text-muted); }\n.thumb-text { color: var(--text-muted); margin-top: 0.15rem; }\n\n.thumb-actions {\n    display: flex;\n    gap: 0.3rem;\n    margin-top: 0.3rem;\n}\n.icon-btn {\n    width: 26px;\n    height: 26px;\n    display: flex;\n    align-items: center;\n    justify-content: center;\n    border: 1px solid var(--border);\n    border-radius: 4px;\n    background: #fff;\n    font-size: 0.85rem;\n    padding: 0;\n}\n.icon-btn:hover { background: var(--accent-soft); }\n.icon-btn:disabled { opacity: 0.4; cursor: not-allowed; }\n\n.group-header {\n    font-size: 0.9rem;\n    margin: 0.75rem 0 0.4rem;\n}\n\n.divider { border: none; border-top: 1px solid var(--border); margin: 1rem 0; }\n\n/* ---------------------------------------------------------------------- */\n/* Export CSV (frontend/export.html, frontend/js/export-ui.js) -- a\n   standalone tab now, not an in-page modal (see export-dialog.js) */\n/* ---------------------------------------------------------------------- */\n\n.export-page {\n    max-width: 1800px;\n    margin: 0 auto;\n    padding: 1.5rem 2rem;\n}\n.export-page h2 { margin-top: 0; }\n\n.export-dialog input[type="text"], .export-dialog input[type="number"] {\n    width: 100%;\n    font-family: inherit;\n    font-size: 0.9rem;\n    padding: 0.4rem 0.5rem;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n}\n\n.segmented { display: flex; border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden; }\n.segmented button {\n    border: none;\n    background: #fff;\n    padding: 0.4rem 0.8rem;\n    font-size: 0.85rem;\n    cursor: pointer;\n    border-right: 1px solid var(--border);\n}\n.segmented button:last-child { border-right: none; }\n.segmented button.active { background: var(--accent); color: #fff; }\n\n/* Everything that used to be three separate rows (query-type segmented\n   control, Query#/Video ID/Frame ID/Change, Cancel/Export) shares one row\n   to save vertical space -- wraps rather than overflowing on a narrow\n   window. */\n.export-topbar {\n    display: flex;\n    align-items: center;\n    gap: 0.6rem;\n    flex-wrap: wrap;\n    margin-bottom: 0.75rem;\n}\n.export-topbar .segmented { flex: none; }\n.export-topbar input#exp-name { flex: 0 0 110px; }\n.export-topbar .export-actions { flex: none; margin-left: auto; }\n\n.export-answer-area { margin: 0.75rem 0; }\n.export-answer-area input[type="text"] { margin-top: 0.5rem; }\n\n.export-answer-list { display: flex; flex-wrap: wrap; gap: 0.6rem; margin-top: 0.5rem; }\n.export-answer-card {\n    position: relative;\n    width: 220px;\n    font-size: 0.78rem;\n    padding: 0.3rem;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    background: #fff;\n}\n.export-answer-card[draggable="true"] { cursor: grab; }\n.export-answer-card .export-remove-btn {\n    position: absolute;\n    top: 2px;\n    right: 2px;\n    /* .thumb-wrap carries z-index:1 unconditionally (style.css\'s "Result\n       cards" section), which -- with no z-index of its own -- would paint\n       above this sibling button despite coming first in the markup. */\n    z-index: 200;\n    width: 20px;\n    height: 20px;\n    font-size: 0.7rem;\n}\n\n.export-preview-area {\n    display: flex;\n    gap: 1.5rem;\n    margin-top: 1rem;\n    flex-wrap: wrap;\n}\n.export-preview-section { flex: 1 1 320px; min-width: 0; }\n.export-preview-header { margin-bottom: 0.4rem; }\n.export-preview-grid { grid-template-columns: repeat(auto-fill, minmax(130px, 1fr)); gap: 0.5rem; }\n.export-preview-grid .thumb-cell { position: relative; font-size: 0.72rem; }\n.export-preview-grid .thumb-caption { font-size: 0.72rem; }\n\n/* Nearby-frames ("Show more") popup -- same top-right export-badge overlay\n   as the export screen\'s preview grids, see dialogs.js\'s openNeighborsDialog. */\n.nbr-grid .thumb-cell { position: relative; }\n.export-add-btn {\n    position: absolute;\n    top: 2px;\n    right: 2px;\n    /* .thumb-wrap:hover jumps to z-index:100 for the hover-zoom effect\n       (style.css\'s "Result cards" section) -- stay above that or a click\n       on a hovered thumbnail\'s add button lands on the zoomed <img>\n       instead. */\n    z-index: 200;\n    width: 20px;\n    height: 20px;\n    font-size: 0.75rem;\n    border: 1px solid var(--border);\n    border-radius: 4px;\n    background: #fff;\n    cursor: pointer;\n}\n.export-add-btn.added { background: var(--accent); border-color: var(--accent); color: #fff; }\n\n.trake-event-list { display: flex; flex-direction: column; gap: 0.5rem; }\n.trake-event-row {\n    position: relative;\n    display: flex;\n    align-items: center;\n    gap: 0.6rem;\n    padding: 0.4rem;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n}\n.trake-event-row[draggable="true"] { cursor: grab; }\n/* Same corner-badge treatment as .export-answer-card\'s remove button --\n   TRAKE events are editable (add/remove/reorder) now, not just a fixed\n   list from a real TRAKE search. */\n.trake-event-row .export-remove-btn {\n    position: absolute;\n    top: 2px;\n    right: 2px;\n    z-index: 200;\n    width: 20px;\n    height: 20px;\n    font-size: 0.7rem;\n}\n.trake-event-row .thumb-wrap, .trake-event-row .thumb-missing { width: 70px; flex: none; }\n.trake-event-fields { flex: 1; }\n\n/* TRAKE curate/cache/merge panel (export-ui.js\'s ensureTrakeSkeleton) --\n   .playback-timer above is shared with the modal playback dialogs, but\n   .playback-layout/.playback-main/.playback-info aren\'t (this panel uses\n   its own .trake-toprow/.trake-main-row/.trake-video-col layout instead,\n   see below); .dialog-box video\'s sizing doesn\'t apply outside a dialog\n   either, so the curation video gets its own equivalent rule here. */\n.export-dialog video {\n    display: block;\n    max-width: 100%;\n    width: auto;\n    height: auto;\n    margin: 0 auto;\n    border-radius: 4px;\n}\n/* Load/switch video + the live video/frame/time readout + "Add current\n   frame" all share one compact row -- the Video ID box and its button are\n   deliberately small (a one-time-per-video action), unlike the wide,\n   prominent Add-frame button (the repeated per-event action). */\n.trake-toprow {\n    display: flex;\n    align-items: center;\n    gap: 0.5rem;\n    flex-wrap: wrap;\n    margin-bottom: 0.75rem;\n}\n.trake-toprow input#trake-load-video { flex: 0 0 170px; padding: 0.3rem 0.5rem; font-size: 0.85rem; }\n.trake-toprow #trake-load-btn { padding: 0.3rem 0.6rem; font-size: 0.8rem; }\n.trake-toprow .playback-timer { margin: 0; }\n\n/* Video preview (left, 65%) and the event list (right, 35%) share one\n   row -- .export-dialog video\'s max-width:100% keeps the video within its\n   65% column regardless of the source video\'s native aspect ratio. */\n.trake-main-row { display: flex; gap: 1.25rem; align-items: flex-start; }\n/* flex-shrink:1 (not 0) so the row\'s gap doesn\'t push the 65/35 split past\n   100% of the container -- both columns give up the same few px, keeping\n   the ratio effectively intact. */\n.trake-video-col { flex: 0 1 65%; min-width: 0; }\n.trake-events-col { flex: 0 1 35%; min-width: 0; }\n\n.trake-cache-row {\n    display: flex;\n    align-items: center;\n    gap: 0.6rem;\n    padding: 0.4rem 0.5rem;\n    margin-bottom: 0.4rem;\n    border: 1px solid var(--border);\n    border-radius: var(--radius);\n    cursor: grab;\n}\n.trake-cache-row .thumb-caption { flex: 1; margin: 0; }\n\n.export-actions { display: flex; gap: 0.5rem; }\n\n/* Video ID/Frame ID/Change -- share the topbar\'s remaining space and\n   collapse to nothing (display:none, set in export-ui.js) while TRAKE is\n   selected -- TRAKE has no single "current export frame" to retarget with\n   Video ID/Frame ID. Query# (also in .export-topbar, see above) stays\n   visible for every type. */\n.export-change-fields { display: flex; gap: 0.5rem; flex: 1 1 260px; min-width: 0; }\n.export-change-fields input { flex: 1; min-width: 0; }\n\n/* ---------------------------------------------------------------------- */\n/* Dialogs */\n/* ---------------------------------------------------------------------- */\n\n.dialog-overlay {\n    position: fixed;\n    inset: 0;\n    background: rgba(0, 0, 0, 0.5);\n    display: flex;\n    align-items: flex-start;\n    justify-content: center;\n    padding: 4rem 1rem;\n    z-index: 1000;\n}\n\n.dialog-box {\n    position: relative;\n    background: #fff;\n    border-radius: 8px;\n    padding: 0.65rem 1rem;\n    width: 100%;\n    max-width: 1100px;\n    max-height: 85vh;\n    overflow-y: auto;\n    box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);\n}\n.dialog-box.wide { max-width: 1100px; }\n\n.dialog-box h3 { margin: 0 2rem 0.5rem 0; } /* right margin clears the close button, which no longer floats */\n\n/* Positioned (not floated) so it works the same whether or not a dialog\n   has a title -- the playback dialogs below have none. */\n.dialog-close {\n    position: absolute;\n    top: 0.6rem;\n    right: 0.75rem;\n    border: none;\n    background: none;\n    font-size: 1.2rem;\n    cursor: pointer;\n    color: var(--text-muted);\n}\n\n/* Title-less playback dialogs (openPlaybackDialog/openTrakePlaybackDialog):\n   video on one side, video_id/frame/timer info beside it instead of above\n   and below -- keeps the dialog shorter so it fits without scrolling. */\n.playback-layout {\n    display: flex;\n    gap: 1rem;\n    align-items: flex-start;\n    margin-top: 1.5rem; /* clears the absolutely-positioned close button, since there\'s no title to do that for us */\n}\n.playback-main {\n    flex: 1 1 auto;\n    min-width: 0;\n    display: flex;\n    flex-direction: column;\n}\n.playback-info { flex: 0 0 240px; }\n.playback-info .thumb-caption { margin-bottom: 0.5rem; }\n.playback-timer { font-family: monospace; font-size: 0.9rem; margin-top: 0.5rem; }\n\n.playback-container {\n    position: relative;\n    width: 100%;\n    min-height: 280px;\n    background: #0f1117;\n    border-radius: 6px;\n    display: flex;\n    flex-direction: column;\n    align-items: center;\n    justify-content: center;\n    overflow: hidden;\n}\n\n.playback-container video {\n    display: block;\n    width: 100%;\n    max-height: 70vh;\n    border-radius: 4px;\n    background: #000;\n    outline: none;\n}\n\n.playback-loading {\n    color: #e2e8f0;\n    font-size: 0.9rem;\n    display: flex;\n    align-items: center;\n    gap: 0.5rem;\n    padding: 2rem;\n}\n\n/* Keyframe Fallback Scrubber / Timeline Slideshow */\n.keyframe-player {\n    width: 100%;\n    display: flex;\n    flex-direction: column;\n    align-items: center;\n    background: #0f1117;\n    border-radius: 6px;\n    padding: 0.5rem;\n}\n.keyframe-player-img-wrap {\n    width: 100%;\n    height: 400px;\n    max-height: 65vh;\n    display: flex;\n    align-items: center;\n    justify-content: center;\n    background: #000;\n    border-radius: 4px;\n    overflow: hidden;\n}\n.keyframe-player-img {\n    max-width: 100%;\n    max-height: 100%;\n    object-fit: contain;\n    border-radius: 2px;\n}\n.keyframe-controls-bar {\n    width: 100%;\n    display: flex;\n    align-items: center;\n    gap: 0.5rem;\n    margin-top: 0.5rem;\n    padding: 0.2rem 0.4rem;\n}\n.keyframe-slider {\n    flex: 1;\n    cursor: pointer;\n}\n.keyframe-ctrl-btn {\n    padding: 0.25rem 0.5rem;\n    font-size: 0.8rem;\n    background: #232733;\n    color: #fff;\n    border: 1px solid #374151;\n    border-radius: 4px;\n    cursor: pointer;\n}\n.keyframe-ctrl-btn:hover { background: #374151; }\n\n.dialog-box video {\n    display: block;\n    max-width: 100%;\n    max-height: 75vh;\n    width: 100%;\n    height: auto;\n    margin: 0 auto;\n    border-radius: 4px;\n    background: #000;\n}\n\n#trake-marker-bar {\n    position: relative;\n    height: 20px;\n    margin: 6px 0 8px;\n    background: rgba(127, 127, 127, 0.15);\n    border-radius: 4px;\n}\n.trake-marker-tick {\n    position: absolute; top: 0; bottom: 0;\n    transform: translateX(-50%);\n    cursor: pointer;\n    background: #e64980;\n    color: white;\n    font-size: 10px;\n    padding: 0 4px;\n    border-radius: 3px;\n    line-height: 20px;\n    user-select: none;\n}\n.trake-marker-tick:hover { filter: brightness(1.2); }\n', encoding="utf-8")

# 10. Cài đặt các thư viện cần thiết
!pip install -q --upgrade pip
!pip install -q fastapi uvicorn python-multipart cachetools "elasticsearch>=8.11,<8.13" multilingual-clip faiss-cpu
!pip install -q transformers timm pillow tqdm pandas numpy
!pip install -q pycloudflared python-dotenv requests ultralytics

print("✅ Bước 1: Mã nguồn đã được vá hoàn chỉnh & Dependencies đã sẵn sàng!")


## 🐘 BƯỚC 2: Cài đặt và khởi chạy Elasticsearch nền (Không cần Docker)

In [ ]:
import os
import time
import requests
import subprocess
from pathlib import Path

print("=" * 60)
print("🐘 CÀI ĐẶT VÀ KHỞI ĐỘNG ELASTICSEARCH NỀN")
print("=" * 60)

# 1. Dọn dẹp sạch tiến trình cũ & file lock
!pkill -9 -f elasticsearch 2>/dev/null || true
!rm -f /opt/elasticsearch-8.11.0/data/node.lock 2>/dev/null || true

# 2. Tải và giải nén Elasticsearch 8.11.0 (nếu chưa tải)
if not Path("/opt/elasticsearch-8.11.0").exists():
    print("📥 Đang tải Elasticsearch 8.11.0 (khoảng 30s)...")
    !wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.11.0-linux-x86_64.tar.gz -O /tmp/es.tar.gz
    !tar -xzf /tmp/es.tar.gz -C /opt/
    !rm -f /tmp/es.tar.gz
    print("✅ Đã giải nén vào /opt/elasticsearch-8.11.0")

# 3. Cấu hình Development Mode an toàn (localhost 127.0.0.1 để bypass bootstrap check)
es_yml = '''cluster.name: kaggle-routing101
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
'''
Path("/opt/elasticsearch-8.11.0/config/elasticsearch.yml").write_text(es_yml, encoding="utf-8")

# Cấu hình Heap 512MB - 1GB
heap_opts_dir = Path("/opt/elasticsearch-8.11.0/config/jvm.options.d")
heap_opts_dir.mkdir(parents=True, exist_ok=True)
(heap_opts_dir / "heap.options").write_text("-Xms512m\n-Xmx1g\n", encoding="utf-8")

# 4. Phân quyền cho esuser và tạo file log trực tiếp trong /kaggle/working/
!useradd -m -s /bin/bash esuser 2>/dev/null || true
!chown -R esuser:esuser /opt/elasticsearch-8.11.0
!touch /kaggle/working/elasticsearch.log && chmod 666 /kaggle/working/elasticsearch.log
!chmod -R 777 /tmp

# 5. Khởi động Elasticsearch nền và lưu log trực tiếp vào /kaggle/working/elasticsearch.log
print("🚀 Đang khởi động tiến trình Elasticsearch...")
es_log_file = open("/kaggle/working/elasticsearch.log", "a")
subprocess.Popen(
    ["su", "-", "esuser", "-c", "/opt/elasticsearch-8.11.0/bin/elasticsearch"],
    stdout=es_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 6. Chờ cổng 9200 sẵn sàng
print("⏳ Đang đợi Elasticsearch sẵn sàng...", end="", flush=True)
es_ready = False
for attempt in range(35):
    try:
        r = requests.get("http://127.0.0.1:9200", timeout=1)
        if r.status_code == 200:
            info = r.json()
            print(f"\n✅ THÀNH CÔNG! Elasticsearch (v{info.get('version', {}).get('number')}) đã sẵn sàng tại http://127.0.0.1:9200")
            print(f"📄 File log Elasticsearch: /kaggle/working/elasticsearch.log")
            es_ready = True
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(1)

if not es_ready:
    print("\n\n❌ Elasticsearch chưa sẵn sàng. Log chi tiết từ /kaggle/working/elasticsearch.log:")
    print("=" * 60)
    !cat /kaggle/working/elasticsearch.log | tail -n 35
    print("=" * 60)


## ⚡ BƯỚC 3: Tự động phát hiện & Ánh xạ đường dẫn siêu nhanh (Deep Pruned Auto-Discovery)

In [ ]:
import os
import csv
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

print("=" * 70)
print("📂 ĐANG QUÉT CẤU TRÚC /kaggle/input (FAST PRUNED DISCOVERY)...")
print("=" * 70)

TARGET_NAMES = {
    "keyframes", "degarr", "videos", "video", "map-keyframes", "map_keyframes",
    "siglib_embed", "siglip_embed", "siglip2_embed",
    "captions", "caption", "caption_embed", "captions_embed",
    "ocr", "summaries", "summary", "summary_embed", "summaries_embed",
    "transcripts", "transcript", "transcript_embed", "transcripts_embed",
    "clip-features-32", "clip_features_32",
    "filtered_object", "filtered_objects", "objects", "object_detection"
}

all_found_dirs = {}

if KAGGLE_INPUT.exists():
    for root, dirs, _ in os.walk(KAGGLE_INPUT):
        root_path = Path(root)
        name_lower = root_path.name.lower()
        
        # Nếu thư mục hiện tại là mục tiêu, ghi nhận và dừng đào sâu
        if name_lower in TARGET_NAMES:
            all_found_dirs[name_lower] = root_path
            dirs.clear()  # Dừng không đào sâu vào các thư mục con
            continue
        
        # Bỏ qua các thư mục con bắt đầu bằng video id
        dirs[:] = [d for d in dirs if not d.startswith(("L0", "L1", "L2", "v_"))]
        
        rel = root_path.relative_to(KAGGLE_INPUT)
        if len(rel.parts) > 0 and len(rel.parts) <= 3:
            indent = "   " * (len(rel.parts) - 1)
            print(f"{indent}└── 📁 {root_path.name}")
else:
    print("⚠️ Không tìm thấy thư mục /kaggle/input/")

print("=" * 70)

# Hàm lấy thư mục tìm thấy theo thứ tự ưu tiên
def get_target(keys: list, fallback: Path) -> Path:
    for k in keys:
        if k.lower() in all_found_dirs:
            return all_found_dirs[k.lower()]
    return fallback

# 1. Từ nguyenthanghuu/aic2026-dataset (Keyframes, Videos, map-keyframes)
KEYFRAMES_DIR = get_target(["keyframes"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/Keyframes")
VIDEOS_DIR = get_target(["degarr", "videos", "video"], KAGGLE_INPUT / "degarr")
MAP_KEYFRAMES_DIR = get_target(["map-keyframes", "map_keyframes"], KAGGLE_INPUT / "datasets/nguyenthanghuu/aic2026-dataset/map-keyframes")

# Tạo thư mục dummy nếu dataset thiếu keyframes/videos raw để StaticFiles không bao giờ lỗi
if not KEYFRAMES_DIR.exists():
    KEYFRAMES_DIR = Path("/kaggle/working/keyframes_dummy")
    KEYFRAMES_DIR.mkdir(parents=True, exist_ok=True)
if not VIDEOS_DIR.exists():
    VIDEOS_DIR = Path("/kaggle/working/videos_dummy")
    VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Từ lmcu000/rrqbundle (Embeddings & CSVs)
SIGLIP_DIR = get_target(["siglib_embed", "siglip_embed", "siglip2_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/siglib_embed")
CAPTION_EMBED_DIR = get_target(["caption_embed", "captions_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/caption_embed")
CAPTIONS_DIR = get_target(["captions", "caption"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/captions")
OCR_DIR = get_target(["ocr"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/ocr")
SUMMARY_EMBED_DIR = get_target(["summary_embed", "summaries_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/summary_embed")
SUMMARIES_DIR = get_target(["summaries", "summary"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/summaries")
TRANSCRIPT_EMBED_DIR = get_target(["transcript_embed", "transcripts_embed"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/transcript_embed")
TRANSCRIPTS_DIR = get_target(["transcripts", "transcript"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/transcripts")

CLIP_DIR = get_target(["clip-features-32", "clip_features_32"], KAGGLE_INPUT / "clip-features-32")

# 3. Object Detection (OD) Filter
FILTERED_OBJECT_DIR = get_target(["filtered_object", "filtered_objects", "objects", "object_detection"], KAGGLE_INPUT / "datasets/lmcu000/rrqbundle/filtered_object")
CLASS_VOCAB_CSV = FILTERED_OBJECT_DIR / "class_vocab.csv" if FILTERED_OBJECT_DIR.exists() else Path("/kaggle/working/class_vocab.csv")

# Nếu có per-video OD CSV nhưng chưa có class_vocab.csv, tự động tạo ngay
if FILTERED_OBJECT_DIR.exists() and not CLASS_VOCAB_CSV.exists():
    print("🔨 Đang tự động tạo class_vocab.csv từ filtered_object/*.csv...")
    names = set()
    for p in FILTERED_OBJECT_DIR.glob("*.csv"):
        if p.name == "class_vocab.csv": continue
        try:
            with open(p, "r", encoding="utf-8", newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames and "class_name" in reader.fieldnames:
                    for row in reader:
                        raw = row.get("class_name")
                        if raw:
                            names.add(" ".join(str(raw).strip().lower().split()))
        except Exception:
            continue
    CLASS_VOCAB_CSV = Path("/kaggle/working/class_vocab.csv")
    with open(CLASS_VOCAB_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_name"])
        for name in sorted(names):
            writer.writerow([name])
    print(f"✅ Đã tạo {len(names)} unique OD classes tại {CLASS_VOCAB_CSV}")

# Đảm bảo thư mục lưu index và summary embed trên ổ ghi được
Path("/kaggle/working/index").mkdir(parents=True, exist_ok=True)
if not SUMMARY_EMBED_DIR.exists():
    SUMMARY_EMBED_DIR = Path("/kaggle/working/summary_embed")
    SUMMARY_EMBED_DIR.mkdir(parents=True, exist_ok=True)

# 4. Ghi đè file backend/config.py với nội dung hoàn chỉnh tự động nhận diện đường dẫn
config_py_content = f'''import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

FETCH_K = 100
DISPLAY_N = 30
RRF_K = 60
NEIGHBOR_WINDOW = 7
TOP_G_DEFAULT = 5

DATASET_MODE = "AIC"
SIGLIP2_MODEL_ID = "google/siglip2-base-patch16-384"

FRAME_SIGLIP2_GLOB = "{SIGLIP_DIR}/*.npy"
FRAME_CLIP_GLOB = "{CLIP_DIR}/*.npy"

ASR_EMBED_DIR = Path("{TRANSCRIPT_EMBED_DIR}")
TRANSCRIPTS_DIR = Path("{TRANSCRIPTS_DIR}")

CAPTIONING_DIR = Path("{CAPTIONS_DIR}")
SIGLIP_CAPTION_DIR = Path("{CAPTION_EMBED_DIR}")

OCR_DIR = Path("{OCR_DIR}")

FILTERED_OBJECT_DIR = Path("{FILTERED_OBJECT_DIR}")
CLASS_VOCAB_CSV = Path("{CLASS_VOCAB_CSV}")

SUMMARY_DIR = Path("{SUMMARIES_DIR}")
SUMMARY_EMBED_DIR = Path("{SUMMARY_EMBED_DIR}")

MAP_KEYFRAMES_DIR = Path("{MAP_KEYFRAMES_DIR}")
THUMBNAIL_ROOT = Path("{KEYFRAMES_DIR}")
VIDEO_DIR = Path("{VIDEOS_DIR}")

INDEX_PREFIX = "routing101"

try:
    SUMMARY_EMBED_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    pass

REPO_ROOT = Path(__file__).resolve().parent.parent
INDEX_DIR = Path("/kaggle/working/index")
PIPELINE_DIR = REPO_ROOT / "pipeline"
ASR_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_asr"
CAPTION_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_caption"
SUMMARY_INDEX_DIR = INDEX_DIR / f"{{INDEX_PREFIX}}_summary"
ASR_INDEX_DIR.mkdir(parents=True, exist_ok=True)
CAPTION_INDEX_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_INDEX_DIR.mkdir(parents=True, exist_ok=True)
SIGLIP_ASR_FAISS = ASR_INDEX_DIR / "siglip_asr_flat_ip.index"
SIGLIP_ASR_META = ASR_INDEX_DIR / "meta_siglip_asr.csv"
SIGLIP_CAPTION_FAISS = CAPTION_INDEX_DIR / "siglip_caption_flat_ip.index"
SIGLIP_CAPTION_META = CAPTION_INDEX_DIR / "meta_siglip_caption.csv"
SIGLIP_SUMMARY_FAISS = SUMMARY_INDEX_DIR / "siglip_summary_flat_ip.index"
SIGLIP_SUMMARY_META = SUMMARY_INDEX_DIR / "meta_siglip_summary.csv"

ES_HOST = "http://127.0.0.1:9200"
ES_INDEX_ASR = "asr_segments"
ES_INDEX_CAPTION = "caption_frames"
ES_INDEX_OCR = "ocr_frames"
ES_INDEX_SUMMARY = "summary_videos"

CPU_BUDGET = max(1, (os.cpu_count() or 4) - 2)

def tune_thread_pools(device: str) -> None:
    import faiss
    import torch
    if device == "cpu":
        torch.set_num_threads(CPU_BUDGET)
    torch.set_num_interop_threads(1)
    faiss.omp_set_num_threads(CPU_BUDGET)
'''

Path("/kaggle/working/Routing101/backend/config.py").write_text(config_py_content, encoding="utf-8")
print("✅ Đã ghi đè cấu hình hoàn hảo vào /kaggle/working/Routing101/backend/config.py!")

# 5. Hiển thị trạng thái
status = lambda p: "✅ SẴN SÀNG" if p.exists() else "⚠️ KHÔNG TÌM THẤY"
print("=" * 70)
print(f"• Keyframes Root         : {status(KEYFRAMES_DIR)} -> {KEYFRAMES_DIR}")
print(f"• Videos Root            : {status(VIDEOS_DIR)} -> {VIDEOS_DIR}")
print(f"• Map-Keyframes CSV      : {status(MAP_KEYFRAMES_DIR)} -> {MAP_KEYFRAMES_DIR}")
print(f"• SigLIP2 Embeddings     : {status(SIGLIP_DIR)} -> {SIGLIP_DIR}")
print(f"• Captions CSV           : {status(CAPTIONS_DIR)} -> {CAPTIONS_DIR}")
print(f"• Caption Embeddings     : {status(CAPTION_EMBED_DIR)} -> {CAPTION_EMBED_DIR}")
print(f"• OCR CSV                : {status(OCR_DIR)} -> {OCR_DIR}")
print(f"• Object Detection (OD)  : {status(FILTERED_OBJECT_DIR)} -> {FILTERED_OBJECT_DIR}")
print(f"• OD Class Vocabulary    : {status(CLASS_VOCAB_CSV)} -> {CLASS_VOCAB_CSV}")
print(f"• Transcripts CSV        : {status(TRANSCRIPTS_DIR)} -> {TRANSCRIPTS_DIR}")
print(f"• Transcript Embeddings  : {status(TRANSCRIPT_EMBED_DIR)} -> {TRANSCRIPT_EMBED_DIR}")
print(f"• Summaries TXT          : {status(SUMMARIES_DIR)} -> {SUMMARIES_DIR}")
print(f"• Summary Embeddings     : {status(SUMMARY_EMBED_DIR)} -> {SUMMARY_EMBED_DIR}")
print(f"• CLIP ViT-B/32 (Opt)    : {status(CLIP_DIR)} -> {CLIP_DIR}")
print("=" * 70)
print("🚀 Cấu hình đường dẫn hoàn tất!")


## 🩺 BƯỚC 4: Kiểm tra nhanh Search Engine (Diagnostics)

In [ ]:
import sys
import time
import importlib
import requests
from pathlib import Path

WORKSPACE_DIR = Path("/kaggle/working/Routing101")
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
%cd /kaggle/working/Routing101

print("=" * 60)
print("🔍 KIỂM TRA HỆ THỐNG VÀ SEARCH ENGINE")
print("=" * 60)

# 1a. Kiểm tra Elasticsearch qua HTTP REST trực tiếp
try:
    resp = requests.get("http://127.0.0.1:9200", timeout=2)
    if resp.status_code == 200:
        info = resp.json()
        print(f"✅ [1/3] Elasticsearch (HTTP 9200): Kết nối OK (Cluster: {info.get('cluster_name')}, v{info.get('version', {}).get('number')})")
    else:
        print(f"⚠️ [1/3] Elasticsearch HTTP returned status {resp.status_code}")
except Exception as e:
    print(f"⚠️ [1/3] Elasticsearch (HTTP 9200) không phản hồi: {e}")
    print("   → Hãy chạy lại Bước 2 trước!")

# 1b. Kiểm tra Elasticsearch Python Client (force-reload module để tránh singleton cũ)
try:
    import backend.config
    importlib.reload(backend.config)
    import backend.es_client
    importlib.reload(backend.es_client)
    from backend.es_client import get_es_client
    
    es = get_es_client(force_new=True)
    host_used = backend.config.ES_HOST
    print(f"   [DEBUG] ES_HOST config = {host_used}")
    
    es_info = es.info()
    cluster = es_info.get('cluster_name', 'unknown')
    version = es_info.get('version', {}).get('number', 'unknown')
    print(f"✅ [2/3] Elasticsearch (Python Client): OK (Cluster: {cluster}, v{version})")
except Exception as e:
    print(f"⚠️ [2/3] Elasticsearch (Python Client): LỖI - {type(e).__name__}: {e}")
    print(f"   → Kiểm tra ES_HOST trong backend/config.py và đảm bảo dùng http://127.0.0.1:9200")

# 2. Kiểm tra tìm kiếm mẫu qua SigLIP2
try:
    from backend.search import keyframe as kf_mod
    t0 = time.time()
    sample_res = kf_mod.search_siglip2_frame("person riding a bicycle", k=5)
    dt = (time.time() - t0) * 1000
    print(f"✅ [3/3] SigLIP2 Vector Search: Thành công ({len(sample_res)} kết quả trong {dt:.1f}ms)")
except Exception as e:
    print(f"⚠️ [3/3] Search test warning: {e}")

print("=" * 60)
print("🎉 Kiểm tra hoàn tất, chuyển sang Bước 5 để mở Web App!")

## 🌐 BƯỚC 5: Khởi chạy Web App & Mở Public URL qua Cloudflare Tunnel

In [ ]:
import subprocess
import time
import re
import os
import requests
from pathlib import Path

%cd /kaggle/working/Routing101

# 1. Tải và cài đặt cloudflared tunnel client (nếu chưa có)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true

# 2. Khởi động FastAPI Backend với Uvicorn
print("🚀 Đang khởi động FastAPI Backend (Routing101)...")
print("   ⏳ Trên CPU, quá trình nạp SigLIP2 + FAISS indexes có thể mất 3-5 phút. Hãy kiên nhẫn!")
backend_log_path = "/kaggle/working/backend.log"
backend_log = open(backend_log_path, "w")
backend_proc = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working/Routing101",
    env=os.environ.copy()
)

# 3. Khởi chạy Cloudflare Tunnel để expose port 8000 ra Internet
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_t = time.time()
while time.time() - start_t < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

# 4. HEALTH CHECK: Đợi đến khi Uvicorn hoàn tất nạp model (timeout 300s = 5 phút)
print("⏳ Đang đợi Backend tải xong dữ liệu và mở cổng 8000...")
backend_ready = False
last_log_pos = 0
for attempt in range(300):
    if backend_proc.poll() is not None:
        print("\n❌ Backend đã dừng đột ngột! Xem log bên dưới:")
        print("-" * 60)
        print(open(backend_log_path).read()[-3000:])
        print("-" * 60)
        break
    try:
        resp = requests.get("http://127.0.0.1:8000/docs", timeout=1)
        if resp.status_code == 200:
            backend_ready = True
            break
    except Exception:
        pass
    # Hiển thị tiến trình từ backend.log
    if attempt > 0 and attempt % 5 == 0:
        try:
            with open(backend_log_path, "r") as lf:
                lf.seek(last_log_pos)
                new_lines = lf.read()
                last_log_pos = lf.tell()
                for l in new_lines.strip().split("\n"):
                    if "[startup]" in l:
                        print(f"   📌 {l.strip()}")
        except Exception:
            pass
    if attempt > 0 and attempt % 30 == 0:
        elapsed = int(time.time() - start_t)
        print(f"   ... đã đợi {elapsed}s, backend vẫn đang nạp dữ liệu...")
    time.sleep(1)

# 5. Hiển thị link truy cập
elapsed_total = int(time.time() - start_t)
if backend_ready and public_url:
    print("\n" + "=" * 70)
    print(f"🎉 WEB APP ROUTING101 ĐÃ KHỞI ĐỘNG XONG SAU {elapsed_total}s!")
    print("=" * 70)
    print(f"🔗 TRUY CẬP GIAO DIỆN TẠI:  {public_url}/app/")
    print(f"📑 API SWAGGER DOCS TẠI  :  {public_url}/docs")
    print("=" * 70 + "\n")
elif not public_url:
    print("❌ Không tạo được Cloudflare Tunnel. Hãy kiểm tra kết nối Internet trong cài đặt Notebook!")
else:
    print(f"⚠️ Backend chưa sẵn sàng sau {elapsed_total}s. Xem log cuối cùng:")
    print("-" * 60)
    print(open(backend_log_path).read()[-2000:])
    print("-" * 60)
    print("💡 Nếu log cho thấy vẫn đang load, hãy chạy Bước 6 để theo dõi tiếp!")

## 📜 BƯỚC 6: Xem Logs Trực Tiếp (Real-time Log Viewer)

Chạy cell dưới đây để theo dõi các truy vấn tìm kiếm, RRF fusion, và thời gian thực thi của backend theo thời gian thực:

In [ ]:
# Xem 50 dòng log gần nhất của backend
!tail -n 50 /kaggle/working/backend.log

# Vòng lặp stream log trực tiếp (Nhấn nút Stop/Interrupt trên thanh công cụ để dừng xem log)
try:
    with open("/kaggle/working/backend.log", "r") as f:
        f.seek(0, 2)  # Seek to end
        print("--- BẮT ĐẦU THEO DÕI LOGS (Nhấn Interrupt để thoát) ---")
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\nĐã dừng theo dõi log. Backend vẫn tiếp tục chạy ngầm.")